# 五個 Feature Models × 五種 ML 方法：Optuna 超參數優化版本

這份 notebook 將原本 `ParameterSampler` 隨機搜尋改成 `Optuna`。

流程：
1. 建立五個 feature model：`Model_W`、`Model_V`、`Model_W_V`、`Model_W_XY`、`Model_W_XY_V`。
2. 使用 rolling train/test split。
3. 對每個 `feature model × ML method` 使用 Optuna 尋找最佳超參數。
4. 輸出類似 learning curve 的 Optuna trial history / best-so-far RMSE 圖。
5. 使用各組最佳超參數訓練 final model，並比較五個模型的測試表現指標。


## 0. 安裝套件

In [ ]:
# 第一次執行才需要
%pip install pandas pyarrow numpy matplotlib tqdm psutil scikit-learn xgboost optuna


## 1. 載入原始 pipeline 函式

這一格保留原 notebook 的資料讀取、欄位偵測、quality filtering、rolling split、metrics 等函式。

In [ ]:
"""
Thesis parquet analysis pipeline WITH GYRO for Data Processing and Evaluation.pdf

Purpose
-------
Run the full analysis for aligned_scan_*_7sensors_full.parquet files and generate
all thesis tables/figures requested in Chapter 4:

Tables 4-1 ~ 4-10
Figures 4-1 ~ 4-11

Expected parquet schema, confirmed from the provided sample file:
- scan_id, point_id_original, point_id_cleaned
- fiber_time, wavelength_nm, IL_Min, MA, x_pos, y_pos
- fiber_start_time, fiber_end_time, filename
- 7 sensor blocks such as N0160_acc_x, N0160_acc_y, N0160_acc_z,
  N0160_acc_mag, optional gyr columns, and *_time_diff_ms

Run example
-----------
python thesis_parquet_analysis_pipeline.py \
  --align-dir "C:/Users/peggy/Desktop/論文實驗/aligned_strict_scans" \
  --output-root "C:/Users/peggy/Desktop/論文實驗/ch4_outputs_with_gyro" \
  --n-experiments 10 \
  --train-size-scans 900

Required packages
-----------------
pip install pandas pyarrow numpy matplotlib xgboost shap tqdm psutil

Notes
-----
1. This script assumes each parquet file is one scan.
2. This WITH-GYRO version uses accelerometer x/y/z/mag plus all available
   gyroscope gyr_x/gyr_y/gyr_z columns by default. Sensors without gyro columns
   remain accelerometer-only instead of causing the whole run to fail.
   Example: 7 sensors where 2 sensors have no gyro -> 7*4 + 5*3 = 43 vibration features.
3. By default this version requires at least one gyro column to exist, so it will
   not silently fall back to accelerometer-only analysis. Use --no-gyro for acc-only
   or --require-all-gyro if every detected sensor must have gyr_x/y/z.
4. The smart-agent section treats G1 residual prediction as Delta_IL_vib.
   For baseline model W+XY, r_W_XY = IL_Min - pred_W_XY.
   Corrected prediction = pred_W_XY + Delta_IL_vib.
   Adjusted observed IL for reporting = IL_Min - Delta_IL_vib.
"""

from __future__ import annotations

import argparse
import gc
import json
import math
import pickle
import re
import textwrap
import warnings
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from tqdm import tqdm
from xgboost import XGBRegressor

try:
    import psutil
except Exception:  # pragma: no cover
    psutil = None

try:
    import shap
except Exception:  # pragma: no cover
    shap = None


# =============================================================================
# Configuration
# =============================================================================


@dataclass
class PipelineConfig:
    align_dir: Path
    output_root: Path

    n_experiments: int = 10
    train_size_scans: int = 900
    test_size_scans: Optional[int] = None
    test_batch_size_scans: int = 50

    target_col: str = "IL_Min"
    vendor_target: float = 0.4
    random_state: int = 42

    # Quality filtering
    remove_wavelength_nm: Optional[float] = 1260.000
    max_abs_time_diff_ms: Optional[float] = None
    drop_missing_required: bool = True

    # Feature settings
    include_model_v: bool = False
    include_gyro: bool = True
    # require_gyro=True means every detected sensor must have gyr_x/y/z.
    # Default False because the real dataset may have sensors without gyro channels.
    require_gyro: bool = False
    # Still require at least one gyro feature when include_gyro=True, avoiding silent acc-only runs.
    require_any_gyro: bool = True

    # Training controls. Use None to train on all rows in the train scans.
    max_train_rows: Optional[int] = None
    max_rows_per_train_scan: Optional[int] = None

    # Plot sampling controls
    plot_sample_per_model_per_fold: int = 60000
    max_scatter_points: int = 200000

    # SHAP controls
    run_shap: bool = True
    shap_groups: Tuple[int, ...] = (1,)
    shap_sample_scans_per_exp: int = 3
    shap_max_rows_per_exp: int = 8000
    shap_max_display: int = 20

    # Smart agent controls
    agent_delta_abs_threshold: float = 0.10
    agent_explain_ratio_threshold: float = 0.30
    agent_max_cases_per_fold: int = 30
    agent_spec_basis: str = "error"  # "error" or "il"

    # XGBoost hyperparameters
    base_n_estimators: int = 500
    base_max_depth: int = 6
    base_learning_rate: float = 0.05
    residual_n_estimators: int = 500
    residual_max_depth: int = 5
    residual_learning_rate: float = 0.03

    @property
    def tables_dir(self) -> Path:
        return self.output_root / "tables"

    @property
    def figures_dir(self) -> Path:
        return self.output_root / "figures"

    @property
    def models_dir(self) -> Path:
        return self.output_root / "models"

    @property
    def intermediate_dir(self) -> Path:
        return self.output_root / "intermediate"

    @property
    def shap_dir(self) -> Path:
        return self.output_root / "shap_analysis"

    @property
    def agent_dir(self) -> Path:
        return self.output_root / "agent_outputs"

    def make_dirs(self) -> None:
        for p in [
            self.output_root,
            self.tables_dir,
            self.figures_dir,
            self.models_dir,
            self.intermediate_dir,
            self.shap_dir,
            self.agent_dir,
        ]:
            p.mkdir(parents=True, exist_ok=True)


# =============================================================================
# General utilities
# =============================================================================


def setup_matplotlib_fonts() -> None:
    """Try to use a CJK-capable font when available; fallback is safe."""
    candidates = [
        "Microsoft JhengHei",
        "Microsoft YaHei",
        "Noto Sans CJK TC",
        "Noto Sans CJK SC",
        "PingFang TC",
        "Heiti TC",
        "Arial Unicode MS",
        "DejaVu Sans",
    ]
    plt.rcParams["font.sans-serif"] = candidates
    plt.rcParams["axes.unicode_minus"] = False


def show_memory(note: str = "") -> None:
    if psutil is None:
        return
    mem = psutil.virtual_memory()
    print(
        f"{note} RAM used={mem.used / 1024**3:.2f} GB, "
        f"available={mem.available / 1024**3:.2f} GB, usage={mem.percent:.1f}%"
    )


def save_pickle(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "wb") as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)


def load_pickle(path: Path):
    with open(path, "rb") as f:
        return pickle.load(f)


def save_csv(df: pd.DataFrame, path: Path, index: bool = False) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(path, index=index, encoding="utf-8-sig")
    print(f"Saved table: {path}")


def save_json(obj, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, ensure_ascii=False, indent=2, default=str)
    print(f"Saved json: {path}")


def read_parquet_safe(path: Path, columns: Optional[List[str]] = None) -> pd.DataFrame:
    try:
        return pd.read_parquet(path, columns=columns)
    except ImportError as e:
        raise ImportError(
            "pandas cannot read parquet because pyarrow/fastparquet is missing. "
            "Please install: pip install pyarrow"
        ) from e


def get_scan_id_from_filename(file: Path) -> Optional[int]:
    file = Path(file)
    m = re.search(r"aligned_scan_(\d+)_7sensors_full\.parquet$", file.name)
    if m:
        return int(m.group(1))
    m = re.search(r"scan[_-]?(\d+)", file.stem)
    if m:
        return int(m.group(1))
    return None


def discover_parquet_files(align_dir: Path) -> List[Path]:
    files = list(Path(align_dir).glob("*.parquet"))
    files = [p for p in files if p.is_file()]
    if not files:
        raise FileNotFoundError(f"No parquet files found in: {align_dir}")

    def key(p: Path):
        sid = get_scan_id_from_filename(p)
        return (sid is None, sid if sid is not None else p.name)

    return sorted(files, key=key)


def make_batches(items: Sequence, batch_size: int) -> Iterable[List]:
    for i in range(0, len(items), batch_size):
        yield list(items[i:i + batch_size])


def downcast_numeric(df: pd.DataFrame) -> pd.DataFrame:
    float64_cols = df.select_dtypes(include=["float64"]).columns
    if len(float64_cols) > 0:
        df[float64_cols] = df[float64_cols].astype(np.float32)
    int64_cols = df.select_dtypes(include=["int64"]).columns
    if len(int64_cols) > 0:
        df[int64_cols] = df[int64_cols].astype(np.int32)
    return df


def sample_df(df: pd.DataFrame, n: int, random_state: int) -> pd.DataFrame:
    if n is None or n <= 0 or len(df) <= n:
        return df.copy()
    return df.sample(n=n, random_state=random_state).copy()


def safe_concat(frames: List[pd.DataFrame]) -> pd.DataFrame:
    frames = [x for x in frames if x is not None and len(x) > 0]
    if not frames:
        return pd.DataFrame()
    return pd.concat(frames, ignore_index=True)


# =============================================================================
# Feature/schema detection
# =============================================================================


def inspect_sample_schema(sample_file: Path) -> pd.DataFrame:
    df = read_parquet_safe(sample_file)
    schema_df = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(t) for t in df.dtypes],
    })
    print("Sample file:", sample_file)
    print("Sample shape:", df.shape)
    print("Sample columns:")
    print(schema_df.to_string(index=False))
    del df
    gc.collect()
    return schema_df


def detect_sensor_ids(columns: Sequence[str]) -> List[str]:
    sensors = set()
    for c in columns:
        m = re.match(r"^([A-Za-z]\d{4})_acc_(x|y|z|mag)$", c)
        if m:
            sensors.add(m.group(1))
    return sorted(sensors)


def build_feature_sets(columns: Sequence[str], cfg: PipelineConfig) -> Tuple[Dict[str, List[str]], Dict[str, str], List[str], List[str]]:
    columns = list(columns)
    required = ["wavelength_nm", "x_pos", "y_pos", cfg.target_col]
    missing = [c for c in required if c not in columns]
    if missing:
        raise ValueError(f"Required columns are missing from parquet schema: {missing}")

    sensor_ids = detect_sensor_ids(columns)
    if not sensor_ids:
        raise ValueError("No accelerometer vibration columns found, e.g., N0160_acc_x.")

    vibration_features = []
    missing_gyro_features = []
    for sensor in sensor_ids:
        for axis in ["x", "y", "z", "mag"]:
            c = f"{sensor}_acc_{axis}"
            if c in columns:
                vibration_features.append(c)
        if cfg.include_gyro:
            for axis in ["x", "y", "z"]:
                c = f"{sensor}_gyr_{axis}"
                if c in columns:
                    vibration_features.append(c)
                else:
                    missing_gyro_features.append(c)

    if cfg.include_gyro and cfg.require_gyro and missing_gyro_features:
        raise ValueError(
            "WITH-GYRO mode is enabled and --require-all-gyro was set, but these gyro columns "
            "are missing from the parquet schema: "
            f"{missing_gyro_features}. Remove --require-all-gyro to use available gyro columns only."
        )

    gyro_features = [c for c in vibration_features if "_gyr_" in c]
    if cfg.include_gyro and cfg.require_any_gyro and len(gyro_features) == 0:
        raise ValueError(
            "WITH-GYRO mode is enabled, but no *_gyr_x/y/z columns were detected. "
            "Please confirm the parquet files contain gyro features, or use --no-gyro for acc-only."
        )
    if cfg.include_gyro and missing_gyro_features and not cfg.require_gyro:
        missing_sensors = sorted({c.split("_gyr_")[0] for c in missing_gyro_features})
        warnings.warn(
            "Some detected sensors do not have complete gyro columns. "
            f"The pipeline will use available gyro columns only. Missing gyro sensors/columns: "
            f"{missing_sensors} / {missing_gyro_features}",
            RuntimeWarning,
        )

    features_w = ["wavelength_nm"]
    features_w_xy = ["wavelength_nm", "x_pos", "y_pos"]
    features_v = vibration_features
    features_w_v = features_w + vibration_features
    features_w_xy_v = features_w_xy + vibration_features

    feature_dict: Dict[str, List[str]] = {
        "Model_W": features_w,
        "Model_W_V": features_w_v,
        "Model_W_XY": features_w_xy,
        "Model_W_XY_V": features_w_xy_v,
    }
    input_label = {
        "Model_W": "wavelength only",
        "Model_W_V": "wavelength + vibration",
        "Model_W_XY": "wavelength + x/y",
        "Model_W_XY_V": "wavelength + x/y + vibration",
    }

    if cfg.include_model_v:
        feature_dict = {
            "Model_W": features_w,
            "Model_V": features_v,
            "Model_W_V": features_w_v,
            "Model_W_XY": features_w_xy,
            "Model_W_XY_V": features_w_xy_v,
        }
        input_label["Model_V"] = "vibration only"

    for model_name, feats in feature_dict.items():
        if len(feats) == 0:
            raise ValueError(f"Feature set is empty for {model_name}")
        missing = [c for c in feats if c not in columns]
        if missing:
            raise ValueError(f"Missing features for {model_name}: {missing}")

    return feature_dict, input_label, vibration_features, sensor_ids


def make_sensor_feature_availability(columns: Sequence[str], sensor_ids: Sequence[str], vibration_features: Sequence[str]) -> pd.DataFrame:
    rows = []
    colset = set(columns)
    featset = set(vibration_features)
    for sensor in sensor_ids:
        acc_cols = [f"{sensor}_acc_{axis}" for axis in ["x", "y", "z", "mag"]]
        gyr_cols = [f"{sensor}_gyr_{axis}" for axis in ["x", "y", "z"]]
        rows.append({
            "Sensor": sensor,
            "Available acc columns": ", ".join([c for c in acc_cols if c in colset]),
            "Available gyro columns": ", ".join([c for c in gyr_cols if c in colset]),
            "Missing gyro columns": ", ".join([c for c in gyr_cols if c not in colset]),
            "Acc feature count used": sum(c in featset for c in acc_cols),
            "Gyro feature count used": sum(c in featset for c in gyr_cols),
            "Has complete gyro": all(c in colset for c in gyr_cols),
            "Has any gyro": any(c in colset for c in gyr_cols),
        })
    return pd.DataFrame(rows)


def get_read_cols(columns: Sequence[str], feature_dict: Dict[str, List[str]], cfg: PipelineConfig) -> List[str]:
    wanted = ["scan_id", cfg.target_col, "wavelength_nm", "x_pos", "y_pos"]
    for feats in feature_dict.values():
        wanted.extend(feats)
    # Keep useful columns if they exist.
    for c in ["fiber_time", "point_id_original", "point_id_cleaned"]:
        if c in columns:
            wanted.append(c)
    # time-diff columns are needed only for quality filtering.
    if cfg.max_abs_time_diff_ms is not None:
        wanted.extend([c for c in columns if c.endswith("_time_diff_ms")])
    wanted = list(dict.fromkeys(wanted))
    return [c for c in wanted if c in columns]


# =============================================================================
# Quality filtering and preprocessing summary
# =============================================================================


def strict_quality_mask(df: pd.DataFrame, cfg: PipelineConfig, required_cols: Sequence[str]) -> Tuple[pd.Series, Dict[str, int]]:
    mask = pd.Series(True, index=df.index)
    stats = {
        "raw_rows": int(len(df)),
        "removed_wavelength_1260_rows": 0,
        "missing_required_rows": 0,
        "invalid_numeric_rows": 0,
        "time_diff_exceed_rows": 0,
    }

    if cfg.remove_wavelength_nm is not None and "wavelength_nm" in df.columns:
        bad_wl = np.isclose(df["wavelength_nm"].astype(float), float(cfg.remove_wavelength_nm))
        stats["removed_wavelength_1260_rows"] = int(np.sum(bad_wl))
        mask &= ~bad_wl

    present_required = [c for c in required_cols if c in df.columns]
    if cfg.drop_missing_required and present_required:
        missing_required = df[present_required].isna().any(axis=1)
        stats["missing_required_rows"] = int(np.sum(missing_required))
        mask &= ~missing_required

    numeric_required = [c for c in present_required if pd.api.types.is_numeric_dtype(df[c])]
    if numeric_required:
        finite_mask = np.isfinite(df[numeric_required].to_numpy(dtype=np.float64)).all(axis=1)
        invalid_numeric = ~finite_mask
        stats["invalid_numeric_rows"] = int(np.sum(invalid_numeric))
        mask &= finite_mask

    if cfg.max_abs_time_diff_ms is not None:
        time_diff_cols = [c for c in df.columns if c.endswith("_time_diff_ms")]
        if time_diff_cols:
            time_ok = df[time_diff_cols].abs().le(float(cfg.max_abs_time_diff_ms)).all(axis=1)
            time_bad = ~time_ok
            stats["time_diff_exceed_rows"] = int(np.sum(time_bad))
            mask &= time_ok

    stats["valid_rows"] = int(mask.sum())
    stats["removed_rows_total_after_union"] = int(len(df) - mask.sum())
    return mask, stats


def profile_and_filter_scans(
    parquet_files: List[Path],
    read_cols: List[str],
    required_cols: Sequence[str],
    cfg: PipelineConfig,
) -> Tuple[pd.DataFrame, List[Path]]:
    rows = []
    valid_files = []

    for file in tqdm(parquet_files, desc="Profiling scans for Table 4-1"):
        scan_id = get_scan_id_from_filename(file)
        try:
            df = read_parquet_safe(file, columns=read_cols)
            if "scan_id" in df.columns and df["scan_id"].notna().any():
                scan_id_data = int(df["scan_id"].dropna().iloc[0])
            else:
                scan_id_data = scan_id

            mask, stats = strict_quality_mask(df, cfg, required_cols)
            valid_scan = bool(stats["valid_rows"] > 0)
            if valid_scan:
                valid_files.append(file)
            row = {
                "file_name": file.name,
                "path": str(file),
                "scan_id_from_file": scan_id,
                "scan_id_in_data": scan_id_data,
                "valid_scan": valid_scan,
                "load_error": "",
            }
            row.update(stats)
            rows.append(row)
            del df
        except Exception as e:
            rows.append({
                "file_name": file.name,
                "path": str(file),
                "scan_id_from_file": scan_id,
                "scan_id_in_data": scan_id,
                "valid_scan": False,
                "load_error": repr(e),
                "raw_rows": 0,
                "valid_rows": 0,
                "removed_wavelength_1260_rows": 0,
                "missing_required_rows": 0,
                "invalid_numeric_rows": 0,
                "time_diff_exceed_rows": 0,
                "removed_rows_total_after_union": 0,
            })
        gc.collect()

    profile_df = pd.DataFrame(rows)
    save_csv(profile_df, cfg.intermediate_dir / "scan_quality_profile.csv")
    return profile_df, valid_files


def make_table_4_1(profile_df: pd.DataFrame, cfg: PipelineConfig, sensor_ids: List[str], vibration_features: List[str]) -> pd.DataFrame:
    raw_scan_count = len(profile_df)
    valid_scan_count = int(profile_df["valid_scan"].sum()) if len(profile_df) else 0
    removed_scan_count = raw_scan_count - valid_scan_count
    total_raw_rows = int(profile_df["raw_rows"].sum()) if "raw_rows" in profile_df else 0
    total_valid_rows = int(profile_df["valid_rows"].sum()) if "valid_rows" in profile_df else 0
    acc_feature_count = sum(1 for c in vibration_features if "_acc_" in c)
    gyro_feature_count = sum(1 for c in vibration_features if "_gyr_" in c)
    gyro_by_sensor = {
        s: sorted([c for c in vibration_features if c.startswith(f"{s}_gyr_")])
        for s in sensor_ids
    }
    sensors_with_complete_gyro = [s for s, cols in gyro_by_sensor.items() if len(cols) == 3]
    sensors_with_partial_gyro = [s for s, cols in gyro_by_sensor.items() if 0 < len(cols) < 3]
    sensors_without_gyro = [s for s, cols in gyro_by_sensor.items() if len(cols) == 0]

    table = pd.DataFrame({
        "Item": [
            "Raw parquet scan files",
            "Valid scans after strict quality filtering",
            "Removed scans",
            "Raw rows before filtering",
            "Valid rows after filtering",
            "Rows removed by wavelength 1260.000 nm rule",
            "Rows removed after union of all filters",
            "Sensor count",
            "Vibration feature count",
            "Accelerometer feature count",
            "Gyroscope feature count",
            "Sensors with complete gyro",
            "Sensors with partial gyro",
            "Sensors without gyro",
            "Vendor/spec threshold for |error|",
            "Max allowed |time_diff_ms|",
            "Target column",
        ],
        "Value": [
            raw_scan_count,
            valid_scan_count,
            removed_scan_count,
            total_raw_rows,
            total_valid_rows,
            int(profile_df.get("removed_wavelength_1260_rows", pd.Series(dtype=int)).sum()),
            int(profile_df.get("removed_rows_total_after_union", pd.Series(dtype=int)).sum()),
            len(sensor_ids),
            len(vibration_features),
            acc_feature_count,
            gyro_feature_count,
            len(sensors_with_complete_gyro),
            len(sensors_with_partial_gyro),
            len(sensors_without_gyro),
            cfg.vendor_target,
            "not applied" if cfg.max_abs_time_diff_ms is None else cfg.max_abs_time_diff_ms,
            cfg.target_col,
        ],
        "Description": [
            "Number of parquet files discovered in input folder.",
            "Files with at least one row remaining after strict filters.",
            "Raw scan files minus valid scan files.",
            "Total rows read from all parquet files before strict filters.",
            "Rows used for model training/testing after strict filters.",
            "Rows where wavelength_nm equals 1260.000 nm.",
            "Rows removed by any strict filtering rule.",
            ", ".join(sensor_ids),
            "Total vibration predictors used by Model_W_V, Model_W_XY_V, residual models and SHAP.",
            "acc_x, acc_y, acc_z and acc_mag from each detected sensor.",
            "Actual gyr_x/gyr_y/gyr_z predictors found and used. Missing gyro columns are not imputed.",
            ", ".join(sensors_with_complete_gyro) if sensors_with_complete_gyro else "none",
            ", ".join(sensors_with_partial_gyro) if sensors_with_partial_gyro else "none",
            ", ".join(sensors_without_gyro) if sensors_without_gyro else "none",
            "Used for Pct_|error|>0.4dB and smart-agent diagnosis.",
            "Optional strict alignment quality filter.",
            "Ground-truth IL column used for prediction.",
        ],
    })
    save_csv(table, cfg.tables_dir / "table_4_1_preprocessing_effective_samples.csv")
    return table


def load_scan_files(
    scan_files: Sequence[Path],
    read_cols: List[str],
    required_cols: Sequence[str],
    cfg: PipelineConfig,
    desc: str,
    max_rows_total: Optional[int] = None,
    max_rows_per_scan: Optional[int] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame]:
    frames = []
    failures = []
    rng = np.random.default_rng(cfg.random_state)

    for file in tqdm(scan_files, desc=desc):
        try:
            df = read_parquet_safe(file, columns=read_cols)
            if "scan_id" not in df.columns:
                sid = get_scan_id_from_filename(file)
                df["scan_id"] = -1 if sid is None else sid
            mask, _ = strict_quality_mask(df, cfg, required_cols)
            df = df.loc[mask].copy()
            if len(df) == 0:
                continue
            if max_rows_per_scan is not None and len(df) > max_rows_per_scan:
                rs = int(rng.integers(0, 2**31 - 1))
                df = df.sample(n=max_rows_per_scan, random_state=rs)
            df = downcast_numeric(df)
            frames.append(df)
        except Exception as e:
            failures.append({"file_name": Path(file).name, "path": str(file), "error": repr(e)})

    if not frames:
        raise ValueError(f"No data loaded for {desc}. First failures: {failures[:3]}")

    out = pd.concat(frames, ignore_index=True)
    if max_rows_total is not None and len(out) > max_rows_total:
        out = out.sample(n=max_rows_total, random_state=cfg.random_state).reset_index(drop=True)
    failures_df = pd.DataFrame(failures)
    del frames
    gc.collect()
    return out, failures_df


# =============================================================================
# Rolling split
# =============================================================================


def make_rolling_time_splits(
    parquet_files: List[Path],
    cfg: PipelineConfig,
) -> List[Dict]:
    n_scans = len(parquet_files)
    if n_scans <= cfg.train_size_scans:
        raise ValueError(
            f"Not enough scans. valid_scans={n_scans}, train_size_scans={cfg.train_size_scans}"
        )
    available = n_scans - cfg.train_size_scans
    test_size = cfg.test_size_scans or available // cfg.n_experiments
    if test_size <= 0:
        raise ValueError("test_size_scans <= 0. Reduce train_size_scans or n_experiments.")

    splits = []
    for exp_id in range(1, cfg.n_experiments + 1):
        train_start_idx = (exp_id - 1) * test_size
        train_end_idx = train_start_idx + cfg.train_size_scans
        test_start_idx = train_end_idx
        test_end_idx = n_scans if exp_id == cfg.n_experiments else test_start_idx + test_size
        if test_start_idx >= n_scans:
            break
        test_end_idx = min(test_end_idx, n_scans)
        train_files = parquet_files[train_start_idx:train_end_idx]
        test_files = parquet_files[test_start_idx:test_end_idx]
        if not train_files or not test_files:
            continue
        splits.append({
            "Experiment": exp_id,
            "train_start_idx": train_start_idx,
            "train_end_idx_exclusive": train_end_idx,
            "test_start_idx": test_start_idx,
            "test_end_idx_exclusive": test_end_idx,
            "train_n_scans": len(train_files),
            "test_n_scans": len(test_files),
            "train_scan_id_start": get_scan_id_from_filename(train_files[0]),
            "train_scan_id_end": get_scan_id_from_filename(train_files[-1]),
            "test_scan_id_start": get_scan_id_from_filename(test_files[0]),
            "test_scan_id_end": get_scan_id_from_filename(test_files[-1]),
            "train_files": train_files,
            "test_files": test_files,
        })
    return splits


def make_table_4_2(splits: List[Dict], cfg: PipelineConfig) -> pd.DataFrame:
    rows = []
    for s in splits:
        rows.append({
            "Experiment": s["Experiment"],
            "Train scan index range": f"{s['train_start_idx']}–{s['train_end_idx_exclusive'] - 1}",
            "Test scan index range": f"{s['test_start_idx']}–{s['test_end_idx_exclusive'] - 1}",
            "Train scan_id range": f"{s['train_scan_id_start']}–{s['train_scan_id_end']}",
            "Test scan_id range": f"{s['test_scan_id_start']}–{s['test_scan_id_end']}",
            "Train scans": s["train_n_scans"],
            "Test scans": s["test_n_scans"],
        })
    table = pd.DataFrame(rows)
    save_csv(table, cfg.tables_dir / "table_4_2_rolling_split_settings.csv")
    return table


# =============================================================================
# Models and metrics
# =============================================================================


def build_xgb_model(cfg: PipelineConfig) -> XGBRegressor:
    return XGBRegressor(
        n_estimators=cfg.base_n_estimators,
        max_depth=cfg.base_max_depth,
        learning_rate=cfg.base_learning_rate,
        subsample=0.8,
        colsample_bytree=0.8,
        eval_metric="rmse",
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=-1,
        random_state=cfg.random_state,
    )


def build_xgb_residual_model(cfg: PipelineConfig) -> XGBRegressor:
    return XGBRegressor(
        n_estimators=cfg.residual_n_estimators,
        max_depth=cfg.residual_max_depth,
        learning_rate=cfg.residual_learning_rate,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_alpha=0.05,
        reg_lambda=2.0,
        eval_metric="rmse",
        objective="reg:squarederror",
        tree_method="hist",
        n_jobs=-1,
        random_state=cfg.random_state,
    )


def metrics_from_error(error: np.ndarray, threshold: float = 0.4) -> Dict[str, float]:
    error = np.asarray(error, dtype=np.float64)
    error = error[np.isfinite(error)]
    if error.size == 0:
        return {
            "N": 0,
            "RMSE": np.nan,
            "MAE": np.nan,
            "Median_AE": np.nan,
            "P90_AE": np.nan,
            "P95_AE": np.nan,
            "P99_AE": np.nan,
            "Pct_|error|>0.4dB": np.nan,
        }
    ae = np.abs(error)
    return {
        "N": int(error.size),
        "RMSE": float(np.sqrt(np.mean(error ** 2))),
        "MAE": float(np.mean(ae)),
        "Median_AE": float(np.median(ae)),
        "P90_AE": float(np.percentile(ae, 90)),
        "P95_AE": float(np.percentile(ae, 95)),
        "P99_AE": float(np.percentile(ae, 99)),
        "Pct_|error|>0.4dB": float(np.mean(ae > threshold) * 100),
    }


def residual_experiment_settings() -> List[Dict]:
    return [
        {
            "Group": 1,
            "Residual Model": "Model_W_XY_V",
            "Base Model": "Model_W_XY",
            "Purpose": "Use W+XY+V to predict the residual of Model_W_XY; isolates vibration-related residual after controlling wavelength and position.",
        },
        {
            "Group": 2,
            "Residual Model": "Model_W_XY_V",
            "Base Model": "Model_W_V",
            "Purpose": "Use W+XY+V to predict the residual of Model_W_V; tests the added value of x/y.",
        },
        {
            "Group": 3,
            "Residual Model": "Model_W_XY",
            "Base Model": "Model_W",
            "Purpose": "Use W+XY to predict the residual of Model_W; tests the added value of x/y over wavelength only.",
        },
        {
            "Group": 4,
            "Residual Model": "Model_W_V",
            "Base Model": "Model_W",
            "Purpose": "Use W+V to predict the residual of Model_W; tests the added value of vibration over wavelength only.",
        },
        {
            "Group": 5,
            "Residual Model": "Model_W_XY_V",
            "Base Model": "Model_W",
            "Purpose": "Use W+XY+V to predict the residual of Model_W; tests the full feature set over wavelength only.",
        },
    ]


def train_base_and_residual_models(
    train_df: pd.DataFrame,
    feature_dict: Dict[str, List[str]],
    input_label: Dict[str, str],
    cfg: PipelineConfig,
    exp_id: int,
) -> Tuple[Dict[str, XGBRegressor], Dict[int, XGBRegressor]]:
    exp_model_dir = cfg.models_dir / f"exp_{exp_id:02d}"
    base_dir = exp_model_dir / "base_models"
    res_dir = exp_model_dir / "residual_models"
    base_dir.mkdir(parents=True, exist_ok=True)
    res_dir.mkdir(parents=True, exist_ok=True)

    base_models: Dict[str, XGBRegressor] = {}
    train_pred_cache: Dict[str, np.ndarray] = {}
    y_train = train_df[cfg.target_col].to_numpy(dtype=np.float32)

    for model_name, features in feature_dict.items():
        print("\n" + "=" * 90)
        print(f"Exp {exp_id}: training base model {model_name} with {len(features)} features")
        show_memory("Before base fit:")
        model = build_xgb_model(cfg)
        model.fit(train_df[features], y_train)
        base_models[model_name] = model
        train_pred_cache[model_name] = model.predict(train_df[features]).astype(np.float32)
        save_pickle(model, base_dir / f"{model_name}.pkl")
        save_pickle({
            "model_name": model_name,
            "features": features,
            "input_feature_label": input_label.get(model_name, ""),
            "target_col": cfg.target_col,
        }, base_dir / f"{model_name}_features.pkl")
        show_memory("After base fit:")

    residual_models: Dict[int, XGBRegressor] = {}
    for setting in residual_experiment_settings():
        group = int(setting["Group"])
        res_model_name = setting["Residual Model"]
        base_model_name = setting["Base Model"]
        if res_model_name not in feature_dict or base_model_name not in base_models:
            print(f"Skip G{group}: feature/model not available.")
            continue
        print("\n" + "=" * 90)
        print(f"Exp {exp_id}: training residual G{group}: {res_model_name} predicts residual of {base_model_name}")
        residual_target = y_train - train_pred_cache[base_model_name]
        res_model = build_xgb_residual_model(cfg)
        res_features = feature_dict[res_model_name]
        res_model.fit(train_df[res_features], residual_target)
        residual_models[group] = res_model
        exp_name = f"G{group}_{res_model_name}_predict_residual_of_{base_model_name}"
        save_pickle(res_model, res_dir / f"{exp_name}.pkl")
        save_pickle({
            "group": group,
            "residual_model_name": res_model_name,
            "base_model_name": base_model_name,
            "features": res_features,
            "target_definition": f"{cfg.target_col} - prediction({base_model_name})",
            "purpose": setting["Purpose"],
        }, res_dir / f"{exp_name}_features.pkl")

    del train_pred_cache
    gc.collect()
    return base_models, residual_models


def sample_error_records(
    errors: np.ndarray,
    labels: np.ndarray,
    n: int,
    random_state: int,
) -> pd.DataFrame:
    if len(errors) == 0 or n <= 0:
        return pd.DataFrame()
    idx = np.arange(len(errors))
    if len(idx) > n:
        rng = np.random.default_rng(random_state)
        idx = rng.choice(idx, size=n, replace=False)
    return pd.DataFrame({
        "label": labels[idx],
        "error": errors[idx].astype(np.float32),
        "abs_error": np.abs(errors[idx]).astype(np.float32),
    })


def evaluate_one_experiment(
    split: Dict,
    feature_dict: Dict[str, List[str]],
    input_label: Dict[str, str],
    read_cols: List[str],
    required_cols: Sequence[str],
    cfg: PipelineConfig,
) -> Dict[str, pd.DataFrame]:
    exp_id = int(split["Experiment"])
    exp_dir = cfg.output_root / f"exp_{exp_id:02d}"
    exp_dir.mkdir(parents=True, exist_ok=True)

    # 1) Train
    train_df, train_failures = load_scan_files(
        split["train_files"],
        read_cols=read_cols,
        required_cols=required_cols,
        cfg=cfg,
        desc=f"Exp {exp_id}: loading train scans",
        max_rows_total=cfg.max_train_rows,
        max_rows_per_scan=cfg.max_rows_per_train_scan,
    )
    if len(train_failures):
        save_csv(train_failures, exp_dir / "train_failed_files.csv")
    print(f"Exp {exp_id}: train_df shape = {train_df.shape}")
    base_models, residual_models = train_base_and_residual_models(
        train_df, feature_dict, input_label, cfg, exp_id
    )
    del train_df
    gc.collect()

    # 2) Test batchwise
    base_error_lists: Dict[str, List[np.ndarray]] = {m: [] for m in base_models}
    residual_before_lists: Dict[int, List[np.ndarray]] = {g: [] for g in residual_models}
    residual_after_lists: Dict[int, List[np.ndarray]] = {g: [] for g in residual_models}

    plot_error_samples = []
    g1_error_samples = []
    delta_scatter_samples = []
    agent_case_chunks = []
    agent_counts = []
    delta_values_for_summary = []
    test_failures_all = []

    settings_by_group = {int(s["Group"]): s for s in residual_experiment_settings()}

    for batch_i, batch_files in enumerate(make_batches(split["test_files"], cfg.test_batch_size_scans), start=1):
        test_df, test_failures = load_scan_files(
            batch_files,
            read_cols=read_cols,
            required_cols=required_cols,
            cfg=cfg,
            desc=f"Exp {exp_id}: loading test batch {batch_i}",
        )
        if len(test_failures):
            test_failures_all.append(test_failures)
        y = test_df[cfg.target_col].to_numpy(dtype=np.float32)

        base_preds: Dict[str, np.ndarray] = {}
        for model_name, model in base_models.items():
            pred = model.predict(test_df[feature_dict[model_name]]).astype(np.float32)
            base_preds[model_name] = pred
            err = y - pred
            base_error_lists[model_name].append(err.astype(np.float32))

        # collect baseline error samples for Figure 4-3
        per_model_sample = max(1, cfg.plot_sample_per_model_per_fold // max(1, len(list(make_batches(split["test_files"], cfg.test_batch_size_scans)))) )
        for model_name in ["Model_W", "Model_W_V", "Model_W_XY", "Model_W_XY_V"]:
            if model_name in base_preds:
                err = y - base_preds[model_name]
                labels = np.array([model_name] * len(err), dtype=object)
                plot_error_samples.append(sample_error_records(err, labels, per_model_sample, cfg.random_state + exp_id + batch_i))

        # residual corrected models
        for group, res_model in residual_models.items():
            setting = settings_by_group[group]
            res_model_name = setting["Residual Model"]
            base_model_name = setting["Base Model"]
            base_pred = base_preds[base_model_name]
            delta = res_model.predict(test_df[feature_dict[res_model_name]]).astype(np.float32)
            corrected_pred = base_pred + delta
            before_err = y - base_pred
            after_err = y - corrected_pred
            residual_before_lists[group].append(before_err.astype(np.float32))
            residual_after_lists[group].append(after_err.astype(np.float32))

            if group == 1:
                # For thesis section 4.5: Delta_IL_vib is G1 residual prediction.
                delta_values_for_summary.append(delta.astype(np.float32))
                labels_before = np.array(["Before: Model_W_XY"] * len(before_err), dtype=object)
                labels_after = np.array(["After: G1 corrected"] * len(after_err), dtype=object)
                g1_error_samples.append(sample_error_records(before_err, labels_before, per_model_sample, cfg.random_state + exp_id + batch_i + 7))
                g1_error_samples.append(sample_error_records(after_err, labels_after, per_model_sample, cfg.random_state + exp_id + batch_i + 11))

                # Figure 4-10 sample: |Delta_IL_vib| vs |r_W_XY|.
                scatter = pd.DataFrame({
                    "Experiment": exp_id,
                    "abs_delta_il_vib": np.abs(delta).astype(np.float32),
                    "abs_r_w_xy": np.abs(before_err).astype(np.float32),
                    "delta_il_vib": delta.astype(np.float32),
                    "r_w_xy": before_err.astype(np.float32),
                })
                if len(scatter) > max(1, cfg.max_scatter_points // max(1, cfg.n_experiments * 2)):
                    scatter = scatter.sample(
                        n=max(1, cfg.max_scatter_points // max(1, cfg.n_experiments * 2)),
                        random_state=cfg.random_state + exp_id + batch_i,
                    )
                delta_scatter_samples.append(scatter)

                if cfg.agent_spec_basis == "il":
                    raw_out = np.abs(y) > cfg.vendor_target
                    corrected_measure = y - delta
                    corrected_in = np.abs(corrected_measure) <= cfg.vendor_target
                    persistent_out = raw_out & (~corrected_in)
                else:
                    raw_out = np.abs(before_err) > cfg.vendor_target
                    corrected_in = np.abs(after_err) <= cfg.vendor_target
                    persistent_out = raw_out & (np.abs(after_err) > cfg.vendor_target)

                ratio = np.abs(delta) / np.maximum(np.abs(before_err), 1e-9)
                suspected_vibration = raw_out & (np.abs(delta) >= cfg.agent_delta_abs_threshold) & (ratio >= cfg.agent_explain_ratio_threshold)
                corrected_back = raw_out & corrected_in

                agent_counts.append({
                    "Experiment": exp_id,
                    "Batch": batch_i,
                    "N": int(len(y)),
                    "Original out-of-spec samples": int(np.sum(raw_out)),
                    "Corrected back in spec": int(np.sum(corrected_back)),
                    "Suspected vibration-related bias": int(np.sum(suspected_vibration)),
                    "Persistent out-of-spec samples": int(np.sum(persistent_out)),
                })

                # representative cases: prefer out-of-spec rows, sorted by |Delta_IL_vib|.
                case_df = pd.DataFrame({
                    "Experiment": exp_id,
                    "scan_id": test_df["scan_id"].to_numpy() if "scan_id" in test_df else np.nan,
                    "wavelength_nm": test_df["wavelength_nm"].to_numpy() if "wavelength_nm" in test_df else np.nan,
                    "x_pos": test_df["x_pos"].to_numpy() if "x_pos" in test_df else np.nan,
                    "y_pos": test_df["y_pos"].to_numpy() if "y_pos" in test_df else np.nan,
                    "IL_raw": y,
                    "Model_W_XY_pred": base_pred,
                    "r_W_XY": before_err,
                    "Delta_IL_vib": delta,
                    "IL_adj_observed_minus_Delta": y - delta,
                    "G1_corrected_pred": corrected_pred,
                    "G1_corrected_error": after_err,
                    "abs_Delta_IL_vib": np.abs(delta),
                    "raw_out_of_spec": raw_out,
                    "corrected_back_in_spec": corrected_back,
                    "suspected_vibration_related_bias": suspected_vibration,
                    "persistent_out_of_spec": persistent_out,
                })
                case_df = case_df[case_df["raw_out_of_spec"]].copy()
                if len(case_df) > 0:
                    case_df["diagnosis_result"] = np.select(
                        [
                            case_df["corrected_back_in_spec"],
                            case_df["suspected_vibration_related_bias"],
                            case_df["persistent_out_of_spec"],
                        ],
                        [
                            "Corrected back within specification",
                            "Suspected vibration-related IL bias",
                            "Persistent out-of-spec after correction",
                        ],
                        default="Out-of-spec; weak vibration evidence",
                    )
                    case_df = case_df.sort_values("abs_Delta_IL_vib", ascending=False).head(cfg.agent_max_cases_per_fold)
                    agent_case_chunks.append(case_df)

        del test_df, y, base_preds
        gc.collect()
        show_memory(f"After Exp {exp_id} test batch {batch_i}:")

    if test_failures_all:
        save_csv(pd.concat(test_failures_all, ignore_index=True), exp_dir / "test_failed_files.csv")

    # 3) Per-experiment metrics
    base_rows = []
    for model_name, chunks in base_error_lists.items():
        err = np.concatenate(chunks) if chunks else np.array([], dtype=np.float32)
        m = metrics_from_error(err, cfg.vendor_target)
        base_rows.append({
            "Experiment": exp_id,
            "Model": model_name,
            "Input / Comparison": input_label.get(model_name, ""),
            **m,
        })
    base_metrics_df = pd.DataFrame(base_rows)
    save_csv(base_metrics_df, exp_dir / "base_model_metrics.csv")

    residual_rows = []
    for group, chunks_before in residual_before_lists.items():
        before_err = np.concatenate(chunks_before) if chunks_before else np.array([], dtype=np.float32)
        after_err = np.concatenate(residual_after_lists[group]) if residual_after_lists[group] else np.array([], dtype=np.float32)
        before_m = metrics_from_error(before_err, cfg.vendor_target)
        after_m = metrics_from_error(after_err, cfg.vendor_target)
        setting = settings_by_group[group]
        rmse_improvement = (before_m["RMSE"] - after_m["RMSE"]) / before_m["RMSE"] * 100 if before_m["RMSE"] and not np.isnan(before_m["RMSE"]) else np.nan
        mae_improvement = (before_m["MAE"] - after_m["MAE"]) / before_m["MAE"] * 100 if before_m["MAE"] and not np.isnan(before_m["MAE"]) else np.nan
        residual_rows.append({
            "Experiment": exp_id,
            "Group": group,
            "Comparison": f"G{group}: {setting['Residual Model']} residual correction for {setting['Base Model']}",
            "Base Model": setting["Base Model"],
            "Residual Model": setting["Residual Model"],
            "Before RMSE": before_m["RMSE"],
            "After RMSE": after_m["RMSE"],
            "RMSE Improvement (%)": rmse_improvement,
            "Before MAE": before_m["MAE"],
            "After MAE": after_m["MAE"],
            "MAE Improvement (%)": mae_improvement,
            "Before P95_AE": before_m["P95_AE"],
            "After P95_AE": after_m["P95_AE"],
            "Before Pct_|error|>0.4dB": before_m["Pct_|error|>0.4dB"],
            "After Pct_|error|>0.4dB": after_m["Pct_|error|>0.4dB"],
            "N": after_m["N"],
        })
    residual_metrics_df = pd.DataFrame(residual_rows)
    save_csv(residual_metrics_df, exp_dir / "residual_correction_metrics.csv")

    # 4) Samples / agent outputs for later tables and plots
    plot_error_df = safe_concat(plot_error_samples)
    if len(plot_error_df):
        plot_error_df.insert(0, "Experiment", exp_id)
        save_csv(plot_error_df, exp_dir / "error_samples_for_fig_4_3.csv")

    g1_error_df = safe_concat(g1_error_samples)
    if len(g1_error_df):
        g1_error_df.insert(0, "Experiment", exp_id)
        save_csv(g1_error_df, exp_dir / "g1_before_after_error_samples_for_fig_4_5.csv")

    scatter_df = safe_concat(delta_scatter_samples)
    if len(scatter_df):
        save_csv(scatter_df, exp_dir / "delta_vib_scatter_samples_for_fig_4_10.csv")

    agent_counts_df = pd.DataFrame(agent_counts)
    if len(agent_counts_df):
        save_csv(agent_counts_df, exp_dir / "agent_counts_by_batch.csv")

    agent_cases_df = safe_concat(agent_case_chunks)
    if len(agent_cases_df):
        save_csv(agent_cases_df, exp_dir / "agent_representative_cases_candidates.csv")

    delta_summary_df = pd.DataFrame()
    if delta_values_for_summary:
        delta_all = np.concatenate(delta_values_for_summary).astype(np.float32)
        abs_delta = np.abs(delta_all)
        delta_summary_df = pd.DataFrame([{
            "Experiment": exp_id,
            "N": int(delta_all.size),
            "Mean": float(np.mean(delta_all)),
            "Std": float(np.std(delta_all, ddof=1)) if delta_all.size > 1 else 0.0,
            "Median": float(np.median(delta_all)),
            "P90_abs": float(np.percentile(abs_delta, 90)),
            "P95_abs": float(np.percentile(abs_delta, 95)),
            "Max_abs": float(np.max(abs_delta)),
        }])
        save_csv(delta_summary_df, exp_dir / "delta_il_vib_summary.csv")

    # Clear model objects
    del base_models, residual_models
    gc.collect()

    return {
        "base_metrics": base_metrics_df,
        "residual_metrics": residual_metrics_df,
        "plot_error_samples": plot_error_df,
        "g1_error_samples": g1_error_df,
        "delta_scatter_samples": scatter_df,
        "agent_counts": agent_counts_df,
        "agent_cases": agent_cases_df,
        "delta_summary": delta_summary_df,
    }


# =============================================================================
# Tables and plots for model evaluation
# =============================================================================


def make_table_4_3(base_cv_results: pd.DataFrame, input_label: Dict[str, str], cfg: PipelineConfig) -> pd.DataFrame:
    thesis_models = ["Model_W", "Model_W_V", "Model_W_XY", "Model_W_XY_V"]
    df = base_cv_results[base_cv_results["Model"].isin(thesis_models)].copy()
    rows = []
    for model, g in df.groupby("Model"):
        rows.append({
            "Model": model,
            "Input features": input_label.get(model, ""),
            "RMSE Mean": g["RMSE"].mean(),
            "RMSE Std": g["RMSE"].std(ddof=1),
            "MAE Mean": g["MAE"].mean(),
            "MAE Std": g["MAE"].std(ddof=1),
            "P95_AE Mean": g["P95_AE"].mean(),
            "P95_AE Std": g["P95_AE"].std(ddof=1),
            "Pct_|error|>0.4dB Mean": g["Pct_|error|>0.4dB"].mean(),
            "Pct_|error|>0.4dB Std": g["Pct_|error|>0.4dB"].std(ddof=1),
        })
    table = pd.DataFrame(rows).sort_values("RMSE Mean")
    save_csv(table, cfg.tables_dir / "table_4_3_baseline_model_performance.csv")
    return table


def make_table_4_4(residual_cv_results: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    agg = residual_cv_results.groupby(["Group", "Comparison", "Base Model", "Residual Model"], as_index=False).agg(
        Before_RMSE_Mean=("Before RMSE", "mean"),
        Before_RMSE_Std=("Before RMSE", "std"),
        After_RMSE_Mean=("After RMSE", "mean"),
        After_RMSE_Std=("After RMSE", "std"),
        RMSE_Improvement_Mean_pct=("RMSE Improvement (%)", "mean"),
        Before_MAE_Mean=("Before MAE", "mean"),
        After_MAE_Mean=("After MAE", "mean"),
        MAE_Improvement_Mean_pct=("MAE Improvement (%)", "mean"),
        Before_P95_AE_Mean=("Before P95_AE", "mean"),
        After_P95_AE_Mean=("After P95_AE", "mean"),
        N_Mean=("N", "mean"),
    ).sort_values("Group")
    save_csv(agg, cfg.tables_dir / "table_4_4_g1_g5_residual_correction_performance.csv")
    return agg


def make_table_4_5(residual_cv_results: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    rows = []
    for group, g in residual_cv_results.groupby("Group"):
        rows.append({
            "Group": group,
            "Comparison": g["Comparison"].iloc[0],
            "RMSE Improvement Mean (%)": g["RMSE Improvement (%)"].mean(),
            "RMSE Improvement Std (%)": g["RMSE Improvement (%)"].std(ddof=1),
            "MAE Improvement Mean (%)": g["MAE Improvement (%)"].mean(),
            "MAE Improvement Std (%)": g["MAE Improvement (%)"].std(ddof=1),
            "Improved folds by RMSE": int((g["RMSE Improvement (%)"] > 0).sum()),
            "Total folds": int(g["Experiment"].nunique()),
        })
    table = pd.DataFrame(rows).sort_values("Group")
    save_csv(table, cfg.tables_dir / "table_4_5_g1_g5_rolling_stability.csv")
    return table


def plot_figure_4_1(cfg: PipelineConfig) -> Path:
    fig, ax = plt.subplots(figsize=(14, 3.4))
    ax.axis("off")
    steps = [
        "IL raw data\n4100 parquet scans",
        "Remove\n1260.000 nm",
        "Build / check\nfiber_time",
        "Vibration backward\nalignment",
        "Strict quality\nfiltering",
        "Aligned valid\ndataset",
        "Rolling\ntime split",
    ]
    x_positions = np.linspace(0.05, 0.95, len(steps))
    for i, (x, txt) in enumerate(zip(x_positions, steps)):
        ax.text(
            x, 0.55, txt,
            ha="center", va="center", fontsize=10,
            bbox=dict(boxstyle="round,pad=0.45", fc="white", ec="black", lw=1.2),
            transform=ax.transAxes,
        )
        if i < len(steps) - 1:
            ax.annotate(
                "", xy=(x_positions[i + 1] - 0.055, 0.55), xytext=(x + 0.055, 0.55),
                arrowprops=dict(arrowstyle="->", lw=1.4), xycoords=ax.transAxes,
            )
    ax.set_title("Figure 4-1. Data cleaning and time-alignment workflow", fontsize=13, pad=20)
    path = cfg.figures_dir / "figure_4_1_data_cleaning_time_alignment_flow.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {path}")
    return path


def plot_figure_4_2(table_4_3: pd.DataFrame, cfg: PipelineConfig) -> Path:
    df = table_4_3.copy().sort_values("Model")
    x = np.arange(len(df))
    width = 0.38
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width / 2, df["RMSE Mean"], width, label="RMSE")
    ax.bar(x + width / 2, df["MAE Mean"], width, label="MAE")
    ax.axhline(cfg.vendor_target, linestyle="--", linewidth=1.2, label=f"Spec threshold {cfg.vendor_target} dB")
    ax.set_xticks(x)
    ax.set_xticklabels(df["Model"], rotation=25, ha="right")
    ax.set_ylabel("Error (dB)")
    ax.set_title("Figure 4-2. Baseline model RMSE and MAE comparison")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    path = cfg.figures_dir / "figure_4_2_baseline_rmse_mae_bar.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {path}")
    return path


def plot_figure_4_3(error_samples: pd.DataFrame, cfg: PipelineConfig) -> Optional[Path]:
    if error_samples.empty:
        return None
    order = [m for m in ["Model_W", "Model_W_V", "Model_W_XY", "Model_W_XY_V"] if m in set(error_samples["label"])]
    data = [error_samples.loc[error_samples["label"] == m, "abs_error"].dropna().to_numpy() for m in order]
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.boxplot(data, labels=order, showfliers=False)
    ax.axhline(cfg.vendor_target, linestyle="--", linewidth=1.3, label=f"|error| = {cfg.vendor_target} dB")
    ax.set_ylabel("Absolute error |IL - prediction| (dB)")
    ax.set_title("Figure 4-3. Baseline absolute error distribution")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    path = cfg.figures_dir / "figure_4_3_baseline_absolute_error_distribution.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {path}")
    return path


def plot_figure_4_4(table_4_4: pd.DataFrame, cfg: PipelineConfig) -> Path:
    df = table_4_4.sort_values("Group")
    labels = [f"G{int(g)}" for g in df["Group"]]
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.bar(labels, df["RMSE_Improvement_Mean_pct"])
    ax.axhline(0, linewidth=1.0)
    ax.set_ylabel("RMSE improvement (%)")
    ax.set_title("Figure 4-4. G1-G5 RMSE improvement comparison")
    ax.grid(axis="y", alpha=0.3)
    for bar, val in zip(bars, df["RMSE_Improvement_Mean_pct"]):
        ax.text(bar.get_x() + bar.get_width() / 2, val, f"{val:.2f}%", ha="center", va="bottom" if val >= 0 else "top", fontsize=9)
    fig.tight_layout()
    path = cfg.figures_dir / "figure_4_4_g1_g5_rmse_improvement.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {path}")
    return path


def plot_figure_4_5(g1_error_samples: pd.DataFrame, cfg: PipelineConfig) -> Optional[Path]:
    if g1_error_samples.empty:
        return None
    order = ["Before: Model_W_XY", "After: G1 corrected"]
    data = [g1_error_samples.loc[g1_error_samples["label"] == m, "error"].dropna().to_numpy() for m in order]
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.boxplot(data, labels=order, showfliers=False)
    ax.axhline(cfg.vendor_target, linestyle="--", linewidth=1.0, label=f"+{cfg.vendor_target} dB")
    ax.axhline(-cfg.vendor_target, linestyle="--", linewidth=1.0, label=f"-{cfg.vendor_target} dB")
    ax.set_ylabel("Signed error / residual (dB)")
    ax.set_title("Figure 4-5. Error distribution before and after G1 correction")
    ax.grid(axis="y", alpha=0.3)
    ax.legend()
    fig.tight_layout()
    path = cfg.figures_dir / "figure_4_5_g1_before_after_error_distribution.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {path}")
    return path


# =============================================================================
# SHAP analysis
# =============================================================================


def feature_major_group(feature: str) -> str:
    if feature == "wavelength_nm":
        return "wavelength"
    if feature in {"x_pos", "y_pos"}:
        return "position"
    return "vibration"


def feature_sensor_group(feature: str) -> str:
    m = re.match(r"^([A-Za-z]\d{4})_", feature)
    if m:
        return m.group(1)
    return feature_major_group(feature)


def get_residual_model_path(cfg: PipelineConfig, exp_id: int, group: int) -> Path:
    setting = {int(s["Group"]): s for s in residual_experiment_settings()}[group]
    exp_name = f"G{group}_{setting['Residual Model']}_predict_residual_of_{setting['Base Model']}"
    return cfg.models_dir / f"exp_{exp_id:02d}" / "residual_models" / f"{exp_name}.pkl"


def sample_test_rows_for_shap(
    test_files: Sequence[Path],
    read_cols: List[str],
    required_cols: Sequence[str],
    cfg: PipelineConfig,
    exp_id: int,
) -> pd.DataFrame:
    rng = np.random.default_rng(cfg.random_state + exp_id)
    files = list(test_files)
    if len(files) > cfg.shap_sample_scans_per_exp:
        idx = rng.choice(np.arange(len(files)), size=cfg.shap_sample_scans_per_exp, replace=False)
        files = [files[i] for i in sorted(idx)]
    df, failures = load_scan_files(
        files,
        read_cols=read_cols,
        required_cols=required_cols,
        cfg=cfg,
        desc=f"Exp {exp_id}: loading SHAP sample",
        max_rows_total=cfg.shap_max_rows_per_exp,
    )
    if len(failures):
        save_csv(failures, cfg.shap_dir / f"exp_{exp_id:02d}_shap_failed_files.csv")
    return df


def compute_shap_for_model(
    model,
    X: pd.DataFrame,
    features: List[str],
    exp_id: int,
    group: int,
    residual_exp_name: str,
) -> Tuple[pd.DataFrame, np.ndarray]:
    if shap is None:
        raise ImportError("shap is not installed. Please install: pip install shap")
    X_feat = X[features].copy()
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(X_feat)
    if hasattr(shap_values, "values"):
        shap_arr = np.asarray(shap_values.values)
    elif isinstance(shap_values, list):
        shap_arr = np.asarray(shap_values[0])
    else:
        shap_arr = np.asarray(shap_values)
    if shap_arr.ndim == 3:
        shap_arr = shap_arr[:, :, 0]
    feature_df = pd.DataFrame({
        "Experiment": exp_id,
        "Group": group,
        "Residual Experiment": residual_exp_name,
        "Feature": features,
        "Feature Group": [feature_major_group(f) for f in features],
        "Sensor": [feature_sensor_group(f) for f in features],
        "MeanAbsSHAP": np.abs(shap_arr).mean(axis=0),
        "MeanSHAP": shap_arr.mean(axis=0),
    }).sort_values("MeanAbsSHAP", ascending=False)
    return feature_df, shap_arr


def run_shap_analysis(
    splits: List[Dict],
    feature_dict: Dict[str, List[str]],
    read_cols: List[str],
    required_cols: Sequence[str],
    cfg: PipelineConfig,
) -> Dict[str, pd.DataFrame]:
    if not cfg.run_shap:
        return {"feature_all": pd.DataFrame(), "group_all": pd.DataFrame(), "sensor_all": pd.DataFrame()}
    if shap is None:
        warnings.warn("SHAP package not installed; skip SHAP tables/figures.")
        return {"feature_all": pd.DataFrame(), "group_all": pd.DataFrame(), "sensor_all": pd.DataFrame()}

    all_feature = []
    all_group = []
    all_sensor = []
    settings = {int(s["Group"]): s for s in residual_experiment_settings()}

    for split in splits:
        exp_id = int(split["Experiment"])
        try:
            shap_df = sample_test_rows_for_shap(split["test_files"], read_cols, required_cols, cfg, exp_id)
        except Exception as e:
            warnings.warn(f"Skip SHAP for exp {exp_id}: {e}")
            continue

        for group in cfg.shap_groups:
            if group not in settings:
                continue
            setting = settings[group]
            model_path = get_residual_model_path(cfg, exp_id, group)
            if not model_path.exists():
                warnings.warn(f"Residual model not found for SHAP: {model_path}")
                continue
            model = load_pickle(model_path)
            features = feature_dict[setting["Residual Model"]]
            exp_name = f"G{group}_{setting['Residual Model']}_predict_residual_of_{setting['Base Model']}"
            feature_df, shap_arr = compute_shap_for_model(model, shap_df, features, exp_id, group, exp_name)
            save_csv(feature_df, cfg.shap_dir / f"exp_{exp_id:02d}_{exp_name}_feature_shap.csv")
            all_feature.append(feature_df)

            group_df = feature_df.groupby(["Experiment", "Group", "Residual Experiment", "Feature Group"], as_index=False).agg(
                TotalMeanAbsSHAP=("MeanAbsSHAP", "sum"),
                TotalMeanSHAP=("MeanSHAP", "sum"),
                FeatureCount=("Feature", "count"),
            )
            group_df["ContributionRatio"] = group_df["TotalMeanAbsSHAP"] / group_df["TotalMeanAbsSHAP"].sum()
            save_csv(group_df, cfg.shap_dir / f"exp_{exp_id:02d}_{exp_name}_feature_group_shap.csv")
            all_group.append(group_df)

            sensor_df = feature_df[feature_df["Feature Group"] == "vibration"].groupby(
                ["Experiment", "Group", "Residual Experiment", "Sensor"], as_index=False
            ).agg(
                SensorMeanAbsSHAP=("MeanAbsSHAP", "sum"),
                SensorMeanSHAP=("MeanSHAP", "sum"),
                FeatureCount=("Feature", "count"),
            )
            if len(sensor_df):
                sensor_df["SensorContributionRatio"] = sensor_df["SensorMeanAbsSHAP"] / sensor_df["SensorMeanAbsSHAP"].sum()
                sensor_df = sensor_df.sort_values("SensorMeanAbsSHAP", ascending=False)
                save_csv(sensor_df, cfg.shap_dir / f"exp_{exp_id:02d}_{exp_name}_sensor_shap.csv")
                all_sensor.append(sensor_df)

            # One SHAP summary plot per exp/group.
            X_feat = shap_df[features].copy()
            plt.figure(figsize=(10, 6))
            shap.summary_plot(shap_arr, X_feat, feature_names=features, show=False, max_display=cfg.shap_max_display)
            fig_path = cfg.shap_dir / f"exp_{exp_id:02d}_{exp_name}_shap_summary_top{cfg.shap_max_display}.png"
            plt.tight_layout()
            plt.savefig(fig_path, dpi=300, bbox_inches="tight")
            plt.close()
            print(f"Saved figure: {fig_path}")

            del model, feature_df, group_df, sensor_df, shap_arr, X_feat
            gc.collect()
        del shap_df
        gc.collect()

    feature_all = safe_concat(all_feature)
    group_all = safe_concat(all_group)
    sensor_all = safe_concat(all_sensor)
    if len(feature_all):
        save_csv(feature_all, cfg.shap_dir / "shap_feature_importance_all_experiments.csv")
    if len(group_all):
        save_csv(group_all, cfg.shap_dir / "shap_feature_group_importance_all_experiments.csv")
    if len(sensor_all):
        save_csv(sensor_all, cfg.shap_dir / "shap_sensor_importance_all_experiments.csv")
    return {"feature_all": feature_all, "group_all": group_all, "sensor_all": sensor_all}


def make_table_4_6(shap_group_all: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    if shap_group_all.empty:
        return pd.DataFrame()
    table = shap_group_all.groupby(["Group", "Residual Experiment", "Feature Group"], as_index=False).agg(
        TotalMeanAbsSHAP=("TotalMeanAbsSHAP", "mean"),
        TotalMeanAbsSHAP_Std=("TotalMeanAbsSHAP", "std"),
        FeatureCount=("FeatureCount", "mean"),
    )
    table["ContributionRatio"] = table.groupby(["Group", "Residual Experiment"])["TotalMeanAbsSHAP"].transform(lambda x: x / x.sum())
    table = table.sort_values(["Group", "TotalMeanAbsSHAP"], ascending=[True, False])
    save_csv(table, cfg.tables_dir / "table_4_6_g1_shap_feature_group_contribution.csv")
    return table


def make_table_4_7(shap_sensor_all: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    if shap_sensor_all.empty:
        return pd.DataFrame()
    table = shap_sensor_all.groupby(["Group", "Residual Experiment", "Sensor"], as_index=False).agg(
        SensorMeanAbsSHAP=("SensorMeanAbsSHAP", "mean"),
        SensorMeanAbsSHAP_Std=("SensorMeanAbsSHAP", "std"),
        SensorMeanSHAP=("SensorMeanSHAP", "mean"),
        FeatureCount=("FeatureCount", "mean"),
    )
    table["ContributionRatio"] = table.groupby(["Group", "Residual Experiment"])["SensorMeanAbsSHAP"].transform(lambda x: x / x.sum())
    table = table.sort_values(["Group", "SensorMeanAbsSHAP"], ascending=[True, False])
    save_csv(table, cfg.tables_dir / "table_4_7_sensor_level_shap_contribution.csv")
    return table


def plot_figure_4_6(table_4_7: pd.DataFrame, cfg: PipelineConfig) -> Optional[Path]:
    if table_4_7.empty:
        return None
    df = table_4_7[table_4_7["Group"] == table_4_7["Group"].min()].sort_values("SensorMeanAbsSHAP", ascending=True)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.barh(df["Sensor"], df["SensorMeanAbsSHAP"], xerr=df["SensorMeanAbsSHAP_Std"].fillna(0), capsize=3)
    ax.set_xlabel("Mean |SHAP| contribution to residual prediction (dB)")
    ax.set_ylabel("Sensor")
    ax.set_title("Figure 4-6. Sensor-level SHAP contribution")
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    path = cfg.figures_dir / "figure_4_6_sensor_level_shap_contribution.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {path}")
    return path


def plot_figure_4_7(shap_feature_all: pd.DataFrame, cfg: PipelineConfig) -> Optional[Path]:
    if shap_feature_all.empty:
        return None
    table = shap_feature_all.groupby(["Feature", "Feature Group"], as_index=False).agg(
        MeanAbsSHAP=("MeanAbsSHAP", "mean"),
        MeanAbsSHAP_Std=("MeanAbsSHAP", "std"),
    ).sort_values("MeanAbsSHAP", ascending=False).head(cfg.shap_max_display)
    save_csv(table, cfg.tables_dir / "figure_4_7_top_features_data.csv")
    df = table.sort_values("MeanAbsSHAP", ascending=True)
    fig, ax = plt.subplots(figsize=(9, 6))
    ax.barh(df["Feature"], df["MeanAbsSHAP"], xerr=df["MeanAbsSHAP_Std"].fillna(0), capsize=3)
    ax.set_xlabel("Mean |SHAP| contribution (dB)")
    ax.set_title("Figure 4-7. SHAP top features bar plot")
    ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    path = cfg.figures_dir / "figure_4_7_shap_top_features_bar.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {path}")
    return path


def copy_or_create_figure_4_8(cfg: PipelineConfig) -> Optional[Path]:
    # Use the first saved SHAP summary plot as thesis Figure 4-8.
    candidates = sorted(cfg.shap_dir.glob("exp_*_G1_*_shap_summary_top*.png"))
    if not candidates:
        return None
    import shutil
    dst = cfg.figures_dir / "figure_4_8_shap_summary_beeswarm.png"
    shutil.copyfile(candidates[0], dst)
    print(f"Saved figure: {dst}")
    return dst


# =============================================================================
# Delta IL vib and smart-agent diagnosis
# =============================================================================


def make_table_4_8(delta_summaries: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    if delta_summaries.empty:
        return pd.DataFrame()
    # Weighted mean/std over folds using available fold-level summary.
    n = delta_summaries["N"].sum()
    weighted_mean = (delta_summaries["Mean"] * delta_summaries["N"]).sum() / n
    # Approximate pooled std from fold means/std.
    pooled_var = (
        ((delta_summaries["N"] - 1) * (delta_summaries["Std"] ** 2)).sum()
        + (delta_summaries["N"] * (delta_summaries["Mean"] - weighted_mean) ** 2).sum()
    ) / max(1, n - 1)
    table = pd.DataFrame([{
        "Delta_IL_vib Mean": weighted_mean,
        "Delta_IL_vib Std": math.sqrt(pooled_var),
        "Delta_IL_vib Median approx": delta_summaries["Median"].median(),
        "Abs Delta_IL_vib P90 approx": delta_summaries["P90_abs"].median(),
        "Abs Delta_IL_vib P95 approx": delta_summaries["P95_abs"].median(),
        "Abs Delta_IL_vib Max": delta_summaries["Max_abs"].max(),
        "N": int(n),
        "Note": "Median/P90/P95 are median-of-fold summaries to avoid storing all rows in memory.",
    }])
    save_csv(table, cfg.tables_dir / "table_4_8_delta_il_vib_statistical_summary.csv")
    return table


def plot_figure_4_9(delta_scatter_samples: pd.DataFrame, cfg: PipelineConfig) -> Optional[Path]:
    if delta_scatter_samples.empty:
        return None
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.hist(delta_scatter_samples["delta_il_vib"].dropna().to_numpy(), bins=80)
    ax.axvline(0, linewidth=1.0)
    ax.set_xlabel("Delta_IL_vib estimated by G1 residual model (dB)")
    ax.set_ylabel("Sample count")
    ax.set_title("Figure 4-9. Distribution of Delta_IL_vib")
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    path = cfg.figures_dir / "figure_4_9_delta_il_vib_distribution.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {path}")
    return path


def plot_figure_4_10(delta_scatter_samples: pd.DataFrame, cfg: PipelineConfig) -> Optional[Path]:
    if delta_scatter_samples.empty:
        return None
    df = delta_scatter_samples.dropna(subset=["abs_delta_il_vib", "abs_r_w_xy"]).copy()
    if len(df) > cfg.max_scatter_points:
        df = df.sample(n=cfg.max_scatter_points, random_state=cfg.random_state)
    fig, ax = plt.subplots(figsize=(7, 6))
    ax.scatter(df["abs_r_w_xy"], df["abs_delta_il_vib"], s=8, alpha=0.35)
    ax.axvline(cfg.vendor_target, linestyle="--", linewidth=1.0, label=f"|r| = {cfg.vendor_target} dB")
    ax.set_xlabel("|r_W_XY| baseline residual magnitude (dB)")
    ax.set_ylabel("|Delta_IL_vib| estimated vibration bias (dB)")
    ax.set_title("Figure 4-10. Delta_IL_vib vs. baseline residual magnitude")
    ax.grid(alpha=0.3)
    ax.legend()
    fig.tight_layout()
    path = cfg.figures_dir / "figure_4_10_delta_il_vib_vs_baseline_residual.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {path}")
    return path


def make_table_4_9(agent_counts: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    if agent_counts.empty:
        return pd.DataFrame()
    cols = [
        "N",
        "Original out-of-spec samples",
        "Corrected back in spec",
        "Suspected vibration-related bias",
        "Persistent out-of-spec samples",
    ]
    sums = agent_counts[cols].sum(numeric_only=True)
    table = pd.DataFrame([{
        "Total evaluated samples": int(sums["N"]),
        "Original out-of-spec samples": int(sums["Original out-of-spec samples"]),
        "Corrected back in spec": int(sums["Corrected back in spec"]),
        "Suspected vibration-related bias": int(sums["Suspected vibration-related bias"]),
        "Persistent out-of-spec samples": int(sums["Persistent out-of-spec samples"]),
        "Spec basis": cfg.agent_spec_basis,
        "Spec threshold": cfg.vendor_target,
        "Delta threshold": cfg.agent_delta_abs_threshold,
        "Explain ratio threshold": cfg.agent_explain_ratio_threshold,
    }])
    save_csv(table, cfg.tables_dir / "table_4_9_smart_agent_diagnosis_summary.csv")
    return table


def make_table_4_10(agent_cases: pd.DataFrame, table_4_7: pd.DataFrame, cfg: PipelineConfig) -> pd.DataFrame:
    if agent_cases.empty:
        return pd.DataFrame()
    cases = agent_cases.copy()
    # Add global top sensors from SHAP Table 4-7. Per-case local SHAP can be added later if desired.
    top_sensors = ""
    if table_4_7 is not None and not table_4_7.empty:
        top_sensors = ", ".join(table_4_7.sort_values("SensorMeanAbsSHAP", ascending=False)["Sensor"].head(3).astype(str).tolist())
    cases["Top contributing sensors"] = top_sensors
    cases = cases.sort_values(["corrected_back_in_spec", "abs_Delta_IL_vib"], ascending=[False, False]).head(20)
    keep = [
        "scan_id",
        "wavelength_nm",
        "x_pos",
        "y_pos",
        "IL_raw",
        "IL_adj_observed_minus_Delta",
        "Delta_IL_vib",
        "r_W_XY",
        "G1_corrected_error",
        "Top contributing sensors",
        "diagnosis_result",
        "Experiment",
    ]
    keep = [c for c in keep if c in cases.columns]
    table = cases[keep].copy()
    save_csv(table, cfg.tables_dir / "table_4_10_smart_agent_representative_cases.csv")
    return table


def plot_figure_4_11(table_4_10: pd.DataFrame, cfg: PipelineConfig) -> Optional[Path]:
    fig, ax = plt.subplots(figsize=(13, 7))
    ax.axis("off")
    ax.set_title("Figure 4-11. Smart agent diagnosis output interface", fontsize=15, pad=20)

    if table_4_10.empty:
        ax.text(0.5, 0.5, "No representative diagnosis cases were generated.", ha="center", va="center", fontsize=12)
    else:
        case = table_4_10.iloc[0]
        left_text = (
            f"Scan ID: {case.get('scan_id', '')}\n"
            f"Wavelength: {float(case.get('wavelength_nm', np.nan)):.3f} nm\n"
            f"Raw IL: {float(case.get('IL_raw', np.nan)):.4f} dB\n"
            f"Adjusted IL: {float(case.get('IL_adj_observed_minus_Delta', np.nan)):.4f} dB\n"
            f"Delta_IL_vib: {float(case.get('Delta_IL_vib', np.nan)):.4f} dB\n"
            f"r_W_XY: {float(case.get('r_W_XY', np.nan)):.4f} dB"
        )
        right_text = (
            f"Top sensors:\n{case.get('Top contributing sensors', '')}\n\n"
            f"Diagnosis:\n{case.get('diagnosis_result', '')}\n\n"
            f"Rule:\n|error| threshold = {cfg.vendor_target} dB\n"
            f"|Delta_IL_vib| threshold = {cfg.agent_delta_abs_threshold} dB"
        )
        ax.text(0.05, 0.72, "Input / estimated quantities", fontsize=12, weight="bold", transform=ax.transAxes)
        ax.text(0.05, 0.35, left_text, fontsize=11, va="top", bbox=dict(boxstyle="round,pad=0.6", fc="white", ec="black"), transform=ax.transAxes)
        ax.text(0.56, 0.72, "Agent diagnosis", fontsize=12, weight="bold", transform=ax.transAxes)
        ax.text(0.56, 0.35, right_text, fontsize=11, va="top", bbox=dict(boxstyle="round,pad=0.6", fc="white", ec="black"), transform=ax.transAxes)
        ax.annotate("", xy=(0.52, 0.48), xytext=(0.43, 0.48), arrowprops=dict(arrowstyle="->", lw=2), xycoords=ax.transAxes)

        # Add a small table preview at the bottom.
        preview = table_4_10.head(5).copy()
        preview_cols = [c for c in ["scan_id", "wavelength_nm", "IL_raw", "IL_adj_observed_minus_Delta", "Delta_IL_vib", "diagnosis_result"] if c in preview.columns]
        preview = preview[preview_cols]
        for c in preview.select_dtypes(include=[np.number]).columns:
            preview[c] = preview[c].map(lambda x: f"{x:.4f}" if pd.notna(x) else "")
        tbl = ax.table(
            cellText=preview.values,
            colLabels=preview.columns,
            loc="bottom",
            cellLoc="center",
            bbox=[0.03, 0.02, 0.94, 0.25],
        )
        tbl.auto_set_font_size(False)
        tbl.set_fontsize(8)

    fig.tight_layout()
    path = cfg.figures_dir / "figure_4_11_smart_agent_diagnosis_interface.png"
    fig.savefig(path, dpi=300, bbox_inches="tight")
    plt.close(fig)
    print(f"Saved figure: {path}")
    return path


# =============================================================================
# Orchestration
# =============================================================================


def run_pipeline(cfg: PipelineConfig) -> None:
    setup_matplotlib_fonts()
    cfg.make_dirs()
    save_json(asdict(cfg), cfg.output_root / "pipeline_config.json")

    parquet_files = discover_parquet_files(cfg.align_dir)
    print(f"Discovered parquet files: {len(parquet_files)}")
    schema_df = inspect_sample_schema(parquet_files[0])
    save_csv(schema_df, cfg.intermediate_dir / "sample_schema.csv")

    columns = schema_df["column"].tolist()
    feature_dict, input_label, vibration_features, sensor_ids = build_feature_sets(columns, cfg)
    required_cols = sorted(set([cfg.target_col, "wavelength_nm", "x_pos", "y_pos"] + vibration_features))
    read_cols = get_read_cols(columns, feature_dict, cfg)

    gyro_availability = make_sensor_feature_availability(columns, sensor_ids, vibration_features)
    save_csv(gyro_availability, cfg.intermediate_dir / "sensor_feature_availability.csv")

    save_json({
        "feature_dict": feature_dict,
        "input_label": input_label,
        "vibration_features": vibration_features,
        "sensor_ids": sensor_ids,
        "read_cols": read_cols,
        "required_cols": required_cols,
        "gyro_availability": gyro_availability.to_dict(orient="records"),
    }, cfg.output_root / "feature_settings.json")

    profile_df, valid_files = profile_and_filter_scans(parquet_files, read_cols, required_cols, cfg)
    if len(valid_files) <= cfg.train_size_scans:
        raise ValueError(
            f"Only {len(valid_files)} valid files after strict filtering; train_size_scans={cfg.train_size_scans}."
        )
    make_table_4_1(profile_df, cfg, sensor_ids, vibration_features)
    plot_figure_4_1(cfg)

    splits = make_rolling_time_splits(valid_files, cfg)
    make_table_4_2(splits, cfg)
    # Save machine-readable split plan without large file objects.
    split_plan_public = []
    for s in splits:
        x = {k: v for k, v in s.items() if k not in {"train_files", "test_files"}}
        x["train_file_names"] = [p.name for p in s["train_files"]]
        x["test_file_names"] = [p.name for p in s["test_files"]]
        split_plan_public.append(x)
    save_json(split_plan_public, cfg.intermediate_dir / "rolling_split_plan.json")

    all_base = []
    all_residual = []
    all_plot_errors = []
    all_g1_errors = []
    all_delta_scatter = []
    all_agent_counts = []
    all_agent_cases = []
    all_delta_summaries = []

    for split in splits:
        exp_id = int(split["Experiment"])
        print("\n" + "#" * 110)
        print(f"START EXPERIMENT {exp_id}")
        print("#" * 110)
        outputs = evaluate_one_experiment(split, feature_dict, input_label, read_cols, required_cols, cfg)
        all_base.append(outputs["base_metrics"])
        all_residual.append(outputs["residual_metrics"])
        all_plot_errors.append(outputs["plot_error_samples"])
        all_g1_errors.append(outputs["g1_error_samples"])
        all_delta_scatter.append(outputs["delta_scatter_samples"])
        all_agent_counts.append(outputs["agent_counts"])
        all_agent_cases.append(outputs["agent_cases"])
        all_delta_summaries.append(outputs["delta_summary"])
        gc.collect()

    base_cv_results = safe_concat(all_base)
    residual_cv_results = safe_concat(all_residual)
    plot_errors = safe_concat(all_plot_errors)
    g1_errors = safe_concat(all_g1_errors)
    delta_scatter = safe_concat(all_delta_scatter)
    agent_counts = safe_concat(all_agent_counts)
    agent_cases = safe_concat(all_agent_cases)
    delta_summaries = safe_concat(all_delta_summaries)

    save_csv(base_cv_results, cfg.intermediate_dir / "base_cv_results_all_experiments.csv")
    save_csv(residual_cv_results, cfg.intermediate_dir / "residual_cv_results_all_experiments.csv")
    if len(plot_errors):
        save_csv(plot_errors, cfg.intermediate_dir / "error_samples_all_experiments.csv")
    if len(g1_errors):
        save_csv(g1_errors, cfg.intermediate_dir / "g1_before_after_error_samples_all_experiments.csv")
    if len(delta_scatter):
        save_csv(delta_scatter, cfg.intermediate_dir / "delta_il_vib_scatter_samples_all_experiments.csv")
    if len(agent_counts):
        save_csv(agent_counts, cfg.intermediate_dir / "agent_counts_all_experiments.csv")
    if len(agent_cases):
        save_csv(agent_cases, cfg.intermediate_dir / "agent_cases_all_experiments.csv")
    if len(delta_summaries):
        save_csv(delta_summaries, cfg.intermediate_dir / "delta_il_vib_summary_by_experiment.csv")

    table_4_3 = make_table_4_3(base_cv_results, input_label, cfg)
    table_4_4 = make_table_4_4(residual_cv_results, cfg)
    make_table_4_5(residual_cv_results, cfg)

    plot_figure_4_2(table_4_3, cfg)
    plot_figure_4_3(plot_errors, cfg)
    plot_figure_4_4(table_4_4, cfg)
    plot_figure_4_5(g1_errors, cfg)

    shap_outputs = run_shap_analysis(splits, feature_dict, read_cols, required_cols, cfg)
    table_4_6 = make_table_4_6(shap_outputs["group_all"], cfg)
    table_4_7 = make_table_4_7(shap_outputs["sensor_all"], cfg)
    plot_figure_4_6(table_4_7, cfg)
    plot_figure_4_7(shap_outputs["feature_all"], cfg)
    copy_or_create_figure_4_8(cfg)

    make_table_4_8(delta_summaries, cfg)
    plot_figure_4_9(delta_scatter, cfg)
    plot_figure_4_10(delta_scatter, cfg)
    make_table_4_9(agent_counts, cfg)
    table_4_10 = make_table_4_10(agent_cases, table_4_7, cfg)
    plot_figure_4_11(table_4_10, cfg)

    print("\nPipeline completed.")
    print("Output root:", cfg.output_root)
    print("Tables:", cfg.tables_dir)
    print("Figures:", cfg.figures_dir)


def parse_args() -> argparse.Namespace:
    p = argparse.ArgumentParser(
        description="Generate thesis Chapter 4 tables/figures from aligned parquet scan files.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter,
    )
    p.add_argument("--align-dir", required=True, type=Path, help="Folder containing aligned_scan_*_7sensors_full.parquet files.")
    p.add_argument("--output-root", required=True, type=Path, help="Output folder for tables, figures, models and intermediates.")
    p.add_argument("--n-experiments", type=int, default=10)
    p.add_argument("--train-size-scans", type=int, default=900)
    p.add_argument("--test-size-scans", type=int, default=None)
    p.add_argument("--test-batch-size-scans", type=int, default=50)
    p.add_argument("--vendor-target", type=float, default=0.4)
    p.add_argument("--random-state", type=int, default=42)
    p.add_argument("--max-abs-time-diff-ms", type=float, default=None)
    p.add_argument("--no-remove-1260", action="store_true", help="Do not remove wavelength_nm == 1260.000 rows.")
    p.add_argument("--include-model-v", action="store_true", help="Train vibration-only Model_V as an extra model.")
    p.add_argument("--no-gyro", action="store_true", help="Disable gyro and use accelerometer-only vibration features.")
    p.add_argument("--require-all-gyro", action="store_true", help="Raise an error if any detected sensor lacks *_gyr_x/y/z columns. Default uses available gyro columns only.")
    p.add_argument("--allow-no-gyro", action="store_true", help="Do not raise an error when include_gyro=True but no gyro columns are found.")
    p.add_argument("--max-train-rows", type=int, default=None, help="Optional cap on total train rows per fold.")
    p.add_argument("--max-rows-per-train-scan", type=int, default=None, help="Optional cap on rows loaded from each train scan.")
    p.add_argument("--skip-shap", action="store_true", help="Skip SHAP analysis.")
    p.add_argument("--shap-sample-scans-per-exp", type=int, default=3)
    p.add_argument("--shap-max-rows-per-exp", type=int, default=8000)
    p.add_argument("--agent-spec-basis", choices=["error", "il"], default="error")
    p.add_argument("--agent-delta-abs-threshold", type=float, default=0.10)
    p.add_argument("--agent-explain-ratio-threshold", type=float, default=0.30)
    p.add_argument("--base-n-estimators", type=int, default=500)
    p.add_argument("--residual-n-estimators", type=int, default=500)
    return p.parse_args()


def main() -> None:
    args = parse_args()
    cfg = PipelineConfig(
        align_dir=args.align_dir,
        output_root=args.output_root,
        n_experiments=args.n_experiments,
        train_size_scans=args.train_size_scans,
        test_size_scans=args.test_size_scans,
        test_batch_size_scans=args.test_batch_size_scans,
        vendor_target=args.vendor_target,
        random_state=args.random_state,
        remove_wavelength_nm=None if args.no_remove_1260 else 1260.000,
        max_abs_time_diff_ms=args.max_abs_time_diff_ms,
        include_model_v=args.include_model_v,
        include_gyro=not args.no_gyro,
        require_gyro=args.require_all_gyro,
        require_any_gyro=not args.allow_no_gyro,
        max_train_rows=args.max_train_rows,
        max_rows_per_train_scan=args.max_rows_per_train_scan,
        run_shap=not args.skip_shap,
        shap_sample_scans_per_exp=args.shap_sample_scans_per_exp,
        shap_max_rows_per_exp=args.shap_max_rows_per_exp,
        agent_spec_basis=args.agent_spec_basis,
        agent_delta_abs_threshold=args.agent_delta_abs_threshold,
        agent_explain_ratio_threshold=args.agent_explain_ratio_threshold,
        base_n_estimators=args.base_n_estimators,
        residual_n_estimators=args.residual_n_estimators,
    )
    run_pipeline(cfg)


# Notebook version: main() is intentionally not auto-called.


## 2. 設定資料夾與實驗參數

In [ ]:
from pathlib import Path

# 請改成你的 aligned parquet 資料夾
ALIGN_DIR = Path(r"C:/Users/peggy/Desktop/論文實驗/aligned_strict_scans")

# 請改成你要放輸出結果的位置
OUTPUT_ROOT = Path(r"C:/Users/peggy/Desktop/論文實驗/ch4_outputs_optuna_5models_ml_benchmark")

cfg = PipelineConfig(
    align_dir=ALIGN_DIR,
    output_root=OUTPUT_ROOT,
    n_experiments=10,
    train_size_scans=900,
    test_size_scans=None,
    test_batch_size_scans=50,
    vendor_target=0.4,
    random_state=42,
    remove_wavelength_nm=1260.000,
    max_abs_time_diff_ms=None,

    # 這份 benchmark 會跑五個 feature models，其中包含 Model_V
    include_model_v=True,
    include_gyro=True,
    require_gyro=False,
    require_any_gyro=True,

    # 若資料太大，可設定例如 300000；None 代表每個 fold 用完整訓練 rows
    max_train_rows=None,
    max_rows_per_train_scan=None,

    # 本 notebook 不跑 SHAP / residual correction；HPO 改用 Optuna
    run_shap=False,
)

cfg


## 3. 五個 Feature Models × 五種 ML 方法：Optuna HPO + 最終表現比較

這一格會針對五個 feature model 與五種 ML 方法分別建立 Optuna study，並輸出 trial 歷程圖、最佳超參數表、final test performance summary。


In [ ]:
# =============================================================================
# Optuna hyperparameter optimization benchmark
#   五個 feature model：
#     Model_W      = wavelength only
#     Model_V      = vibration only
#     Model_W_V    = wavelength + vibration
#     Model_W_XY   = wavelength + x/y
#     Model_W_XY_V = wavelength + x/y + vibration
#   五種機器學習方法：Ridge / RandomForest / ExtraTrees / HistGradientBoosting / XGBoost
#   輸出：
#     1) Optuna trial history，類似 learning curve 的最佳 RMSE 追蹤圖
#     2) 每個模型套用最佳超參數後，在 rolling test set 上的指標比較
# =============================================================================

import gc
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from sklearn.base import clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    HistGradientBoostingRegressor,
)
from sklearn.metrics import r2_score
from xgboost import XGBRegressor


# -----------------------------
# 可調參數
# -----------------------------
# 五個 thesis feature models。名稱已改成直觀版。
MODEL_NAMES_TO_RUN = [
    "Model_W",
    "Model_V",
    "Model_W_V",
    "Model_W_XY",
    "Model_W_XY_V",
]

# 五種 ML 方法。若想先快速測試，可只留 1~2 個方法。
METHOD_NAMES_TO_RUN = [
    "Ridge",
    "RandomForest",
    "ExtraTrees",
    "HistGradientBoosting",
    "XGBoost",
]

# 每個「feature model × ML method」跑幾次 Optuna trial。
# 快速測試可設 10~20；正式實驗建議 40~100。
N_OPTUNA_TRIALS = 40

# HPO 階段只用 train window 中切出的 tuning subset，以節省時間。
# None 代表不限制 rows，但五個模型 × 五種方法會很慢。
HYPEROPT_MAX_TRAIN_ROWS = 120_000
HYPEROPT_MAX_VALID_ROWS = 60_000
HYPEROPT_VALID_SCAN_RATIO = 0.20

# 最終模型訓練資料量。
# None = 不限制；資料大時建議保留上限，避免 RandomForest/ExtraTrees MemoryError。
FINAL_MAX_TRAIN_ROWS = 200_000

# True：只在第一個 rolling split 的 train data 做一次 HPO，之後所有 folds 共用 best params。
# False：每個 rolling split 都重新做 HPO，比較嚴謹但會非常久。
TUNE_ON_FIRST_SPLIT_ONLY = True

# 要跑幾個 rolling experiments；None = 使用 cfg.n_experiments 產生的全部 folds。
# 快速測試建議先設 1；正式跑可以改 None。
MAX_EXPERIMENTS_TO_RUN = 1

# Optuna study 儲存方式：
# None 代表存在記憶體中；若想可續跑，可改成例如 "sqlite:///optuna_wxyv.db"。
OPTUNA_STORAGE = None
OPTUNA_LOAD_IF_EXISTS = True

# 若 Notebook 上方 cfg.include_model_v 原本是 False，這裡強制打開，確保會建立 Model_V。
cfg.include_model_v = True

BENCHMARK_DIR = cfg.output_root / "optuna_5models_ml_benchmark"
MODEL_DIR = BENCHMARK_DIR / "models"
FIGURE_DIR = BENCHMARK_DIR / "figures"
BENCHMARK_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
cfg.make_dirs()

optuna.logging.set_verbosity(optuna.logging.WARNING)


# -----------------------------
# 小工具
# -----------------------------
def jsonable_params(params):
    out = {}
    for k, v in params.items():
        if hasattr(v, "item"):
            v = v.item()
        out[k] = v
    return out


def regression_metrics(y_true, y_pred, threshold=0.4):
    y_true = np.asarray(y_true, dtype=np.float64)
    y_pred = np.asarray(y_pred, dtype=np.float64)
    err = y_true - y_pred
    m = metrics_from_error(err, threshold=threshold)
    finite = np.isfinite(y_true) & np.isfinite(y_pred)
    m["R2"] = float(r2_score(y_true[finite], y_pred[finite])) if finite.sum() > 1 else np.nan

    # 這裡補 MAPE，方便跟 RMSE / MAE 一起比較。
    denom = np.maximum(np.abs(y_true[finite]), 1e-6)
    m["MAPE"] = float(np.mean(np.abs(y_true[finite] - y_pred[finite]) / denom) * 100) if finite.sum() else np.nan
    return m


def method_specs(random_state: int):
    """五種 ML 方法與各自 Optuna search space。"""
    return {
        "Ridge": {
            "estimator": Pipeline([
                ("scaler", StandardScaler()),
                ("model", Ridge()),
            ]),
            "suggest": lambda trial: {
                "model__alpha": trial.suggest_float("model__alpha", 1e-3, 1e3, log=True),
                "model__fit_intercept": trial.suggest_categorical("model__fit_intercept", [True]),
            },
        },
        "RandomForest": {
            "estimator": RandomForestRegressor(
                n_jobs=4,
                random_state=random_state,
                bootstrap=True,
            ),
            "suggest": lambda trial: {
                "n_estimators": trial.suggest_int("n_estimators", 100, 300),
                "max_depth": trial.suggest_int("max_depth", 6, 28),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 3, 15),
                "max_features": trial.suggest_float("max_features", 0.3, 0.7),
                "max_samples": trial.suggest_float("max_samples", 0.5, 0.8),
            },
        },
        "ExtraTrees": {
            "estimator": ExtraTreesRegressor(
                n_jobs=4,
                random_state=random_state,
            ),
            "suggest": lambda trial: {
                "n_estimators": trial.suggest_int("n_estimators", 100, 300),
                "max_depth": trial.suggest_int("max_depth", 6, 28),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 3, 15),
                "max_features": trial.suggest_float("max_features", 0.3, 0.7),
            },
        },
        "HistGradientBoosting": {
            "estimator": HistGradientBoostingRegressor(
                random_state=random_state,
                loss="squared_error",
                early_stopping=True,
                validation_fraction=0.10,
            ),
            "suggest": lambda trial: {
                "max_iter": trial.suggest_int("max_iter", 150, 800),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "max_leaf_nodes": trial.suggest_int("max_leaf_nodes", 15, 80),
                "min_samples_leaf": trial.suggest_int("min_samples_leaf", 15, 150),
                "l2_regularization": trial.suggest_float("l2_regularization", 1e-3, 2.0, log=True),
            },
        },
        "XGBoost": {
            "estimator": XGBRegressor(
                objective="reg:squarederror",
                eval_metric="rmse",
                tree_method="hist",
                n_jobs=-1,
                random_state=random_state,
                verbosity=0,
            ),
            "suggest": lambda trial: {
                "n_estimators": trial.suggest_int("n_estimators", 150, 800),
                "max_depth": trial.suggest_int("max_depth", 3, 10),
                "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.2, log=True),
                "subsample": trial.suggest_float("subsample", 0.6, 1.0),
                "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
                "min_child_weight": trial.suggest_int("min_child_weight", 1, 15),
                "reg_lambda": trial.suggest_float("reg_lambda", 0.1, 10.0, log=True),
                "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 1.0, log=True),
            },
        },
    }


def make_tune_train_valid_files(train_files, valid_ratio=0.20):
    """在 rolling train window 內，以時間順序切出最後一段 scans 當 validation，避免用到 test scans。"""
    n_valid = max(1, int(round(len(train_files) * valid_ratio)))
    n_valid = min(n_valid, len(train_files) - 1)
    tune_train_files = train_files[:-n_valid]
    tune_valid_files = train_files[-n_valid:]
    return tune_train_files, tune_valid_files


def tune_one_model_method(model_name, method_name, spec, features, X_train_all, y_train, X_valid_all, y_valid, random_state, hpo_exp_id):
    """對單一 feature model × ML method 執行 Optuna tuning。目標函數：Validation RMSE 最小。"""
    X_train = X_train_all[features]
    X_valid = X_valid_all[features]

    study_name = f"exp{hpo_exp_id:02d}_{model_name}_{method_name}"
    sampler = TPESampler(seed=random_state)
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=0)
    study = optuna.create_study(
        study_name=study_name,
        direction="minimize",
        sampler=sampler,
        pruner=pruner,
        storage=OPTUNA_STORAGE,
        load_if_exists=OPTUNA_LOAD_IF_EXISTS,
    )

    def objective(trial):
        params = spec["suggest"](trial)
        t0 = time.time()

        model = clone(spec["estimator"])
        model.set_params(**params)
        model.fit(X_train, y_train)

        pred = model.predict(X_valid)
        metrics = regression_metrics(y_valid, pred, threshold=cfg.vendor_target)
        elapsed = time.time() - t0

        # Trial user attrs 會被 trials_dataframe 帶出，方便後續分析。
        trial.set_user_attr("HPO_Experiment", int(hpo_exp_id))
        trial.set_user_attr("Model", model_name)
        trial.set_user_attr("Method", method_name)
        trial.set_user_attr("Feature_Count", len(features))
        trial.set_user_attr("Elapsed_sec", elapsed)
        for k, v in metrics.items():
            trial.set_user_attr(k, v)

        print(
            f"[Optuna] Exp {hpo_exp_id:02d} | {model_name:13s} | {method_name:22s} "
            f"trial {trial.number + 1:03d}/{N_OPTUNA_TRIALS:03d} "
            f"RMSE={metrics['RMSE']:.6f}, MAE={metrics['MAE']:.6f}, "
            f"P95_AE={metrics['P95_AE']:.6f}, time={elapsed:.1f}s"
        )

        del model, pred
        gc.collect()
        return metrics["RMSE"]

    # 若使用 sqlite 且已經跑過，可避免重複補滿超過 N_OPTUNA_TRIALS。
    remaining_trials = max(0, N_OPTUNA_TRIALS - len(study.trials))
    if remaining_trials > 0:
        study.optimize(objective, n_trials=remaining_trials, gc_after_trial=True, show_progress_bar=False)

    best_params = jsonable_params(study.best_params)
    trials_df = study.trials_dataframe(attrs=("number", "value", "params", "user_attrs", "state"))

    # 整理欄位名稱，讓輸出的 csv 比較好讀。
    rename_map = {
        "number": "Trial",
        "value": "RMSE",
        "state": "State",
        "user_attrs_HPO_Experiment": "HPO_Experiment",
        "user_attrs_Model": "Model",
        "user_attrs_Method": "Method",
        "user_attrs_Feature_Count": "Feature_Count",
        "user_attrs_Elapsed_sec": "Elapsed_sec",
        "user_attrs_MAE": "MAE",
        "user_attrs_Median_AE": "Median_AE",
        "user_attrs_P90_AE": "P90_AE",
        "user_attrs_P95_AE": "P95_AE",
        "user_attrs_P99_AE": "P99_AE",
        "user_attrs_Pct_|error|>0.4dB": "Pct_|error|>0.4dB",
        "user_attrs_R2": "R2",
        "user_attrs_MAPE": "MAPE",
        "user_attrs_N": "N",
    }
    trials_df = trials_df.rename(columns=rename_map)
    trials_df["Trial"] = trials_df["Trial"].astype(int) + 1

    # 如果某些欄位因為 study 續跑/失敗 trial 未帶出，補上基本資訊。
    trials_df["HPO_Experiment"] = trials_df.get("HPO_Experiment", hpo_exp_id)
    trials_df["Model"] = trials_df.get("Model", model_name)
    trials_df["Method"] = trials_df.get("Method", method_name)
    trials_df["Feature_Count"] = trials_df.get("Feature_Count", len(features))
    trials_df["Best_RMSE_so_far"] = trials_df["RMSE"].cummin()

    print(f"[BEST] Exp {hpo_exp_id:02d} | {model_name} | {method_name}: RMSE={study.best_value:.6f}, params={best_params}")
    return best_params, trials_df, study


def tune_all_model_methods_for_split(split, read_cols, required_cols, feature_dict, random_state):
    tune_train_files, tune_valid_files = make_tune_train_valid_files(
        split["train_files"],
        valid_ratio=HYPEROPT_VALID_SCAN_RATIO,
    )
    hpo_exp_id = int(split["Experiment"])

    tune_train_df, train_failures = load_scan_files(
        tune_train_files,
        read_cols=read_cols,
        required_cols=required_cols,
        cfg=cfg,
        desc=f"HPO Exp {hpo_exp_id}: loading tuning train scans",
        max_rows_total=HYPEROPT_MAX_TRAIN_ROWS,
        max_rows_per_scan=cfg.max_rows_per_train_scan,
    )
    tune_valid_df, valid_failures = load_scan_files(
        tune_valid_files,
        read_cols=read_cols,
        required_cols=required_cols,
        cfg=cfg,
        desc=f"HPO Exp {hpo_exp_id}: loading tuning validation scans",
        max_rows_total=HYPEROPT_MAX_VALID_ROWS,
        max_rows_per_scan=cfg.max_rows_per_train_scan,
    )

    if len(train_failures):
        save_csv(train_failures, BENCHMARK_DIR / f"hpo_exp_{hpo_exp_id:02d}_train_failed_files.csv")
    if len(valid_failures):
        save_csv(valid_failures, BENCHMARK_DIR / f"hpo_exp_{hpo_exp_id:02d}_valid_failed_files.csv")

    y_train = tune_train_df[cfg.target_col].to_numpy(dtype=np.float32)
    y_valid = tune_valid_df[cfg.target_col].to_numpy(dtype=np.float32)

    print("HPO tuning train shape:", tune_train_df.shape)
    print("HPO validation shape:", tune_valid_df.shape)

    specs = method_specs(random_state)
    best_params_by_combo = {}
    all_trials = []
    all_studies = {}

    for model_name in MODEL_NAMES_TO_RUN:
        features = feature_dict[model_name]
        for method_name in METHOD_NAMES_TO_RUN:
            print("\n" + "=" * 120)
            print(f"Optuna tuning: {model_name} × {method_name}")
            print("=" * 120)

            best_params, trials_df, study = tune_one_model_method(
                model_name=model_name,
                method_name=method_name,
                spec=specs[method_name],
                features=features,
                X_train_all=tune_train_df,
                y_train=y_train,
                X_valid_all=tune_valid_df,
                y_valid=y_valid,
                random_state=random_state,
                hpo_exp_id=hpo_exp_id,
            )
            best_params_by_combo[(model_name, method_name)] = best_params
            all_trials.append(trials_df)
            all_studies[(model_name, method_name)] = study

    hpo_results = pd.concat(all_trials, ignore_index=True)

    del tune_train_df, tune_valid_df, y_train, y_valid
    gc.collect()

    return best_params_by_combo, hpo_results, all_studies


def plot_optuna_history(hpo_results_df, save_dir):
    """類似 learning curve：每個 trial 的 RMSE，以及截至該 trial 的最佳 RMSE。"""
    save_dir.mkdir(parents=True, exist_ok=True)

    # 1) 每個 feature model 一張圖：不同方法的 best-so-far RMSE。
    for model_name, sub_model in hpo_results_df.groupby("Model"):
        fig, ax = plt.subplots(figsize=(9, 5))
        for method_name, sub in sub_model.groupby("Method"):
            sub = sub.sort_values("Trial")
            ax.plot(sub["Trial"], sub["Best_RMSE_so_far"], marker="o", linewidth=1.5, markersize=3, label=method_name)
        ax.set_title(f"Optuna Optimization History: {model_name}")
        ax.set_xlabel("Trial")
        ax.set_ylabel("Best validation RMSE so far")
        ax.grid(alpha=0.3)
        ax.legend()
        fig.tight_layout()
        fig.savefig(save_dir / f"optuna_history_{model_name}.png", dpi=150)
        plt.show()

    # 2) 每個方法一張圖：不同 feature model 的 best-so-far RMSE。
    for method_name, sub_method in hpo_results_df.groupby("Method"):
        fig, ax = plt.subplots(figsize=(9, 5))
        for model_name, sub in sub_method.groupby("Model"):
            sub = sub.sort_values("Trial")
            ax.plot(sub["Trial"], sub["Best_RMSE_so_far"], marker="o", linewidth=1.5, markersize=3, label=model_name)
        ax.set_title(f"Optuna Optimization History: {method_name}")
        ax.set_xlabel("Trial")
        ax.set_ylabel("Best validation RMSE so far")
        ax.grid(alpha=0.3)
        ax.legend()
        fig.tight_layout()
        fig.savefig(save_dir / f"optuna_history_{method_name}.png", dpi=150)
        plt.show()


def plot_final_metric_comparison(final_df, save_dir, tag="all_experiments"):
    """畫出五個 feature model 套用最佳超參數後的測試表現比較。"""
    save_dir.mkdir(parents=True, exist_ok=True)
    metrics_to_plot = ["RMSE", "MAE", "MAPE", "P95_AE", "Pct_|error|>0.4dB"]

    # 每個 ML method：比較五個 feature models。
    for method_name, sub_method in final_df.groupby("Method"):
        plot_df = (
            sub_method
            .groupby("Model", as_index=False)[metrics_to_plot]
            .mean(numeric_only=True)
            .set_index("Model")
            .reindex(MODEL_NAMES_TO_RUN)
        )
        for metric in metrics_to_plot:
            fig, ax = plt.subplots(figsize=(8, 4.5))
            values = plot_df[metric].dropna()
            bars = ax.bar(values.index, values.values)
            ax.set_title(f"{method_name}: five model comparison by {metric}")
            ax.set_xlabel("Feature model")
            ax.set_ylabel(metric)
            ax.tick_params(axis="x", rotation=30)
            for bar, v in zip(bars, values.values):
                ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{v:.4f}", ha="center", va="bottom", fontsize=8)
            ax.grid(axis="y", alpha=0.3)
            fig.tight_layout()
            fig.savefig(save_dir / f"final_{tag}_{method_name}_{metric.replace('|','abs').replace('>','gt')}.png", dpi=150)
            plt.show()

    # 每個 feature model 的最佳 ML method，整理成一張總比較圖。
    best_per_model = (
        final_df
        .groupby(["Model", "Method"], as_index=False)[metrics_to_plot]
        .mean(numeric_only=True)
        .sort_values(["Model", "RMSE"])
        .groupby("Model", as_index=False)
        .first()
        .set_index("Model")
        .reindex(MODEL_NAMES_TO_RUN)
        .reset_index()
    )
    save_csv(best_per_model, save_dir / f"final_{tag}_best_method_per_model.csv")

    fig, ax = plt.subplots(figsize=(8, 4.5))
    values = best_per_model.set_index("Model")["RMSE"].dropna()
    bars = ax.bar(values.index, values.values)
    ax.set_title("Best ML method per feature model: RMSE comparison")
    ax.set_xlabel("Feature model")
    ax.set_ylabel("Test RMSE")
    ax.tick_params(axis="x", rotation=30)
    for bar, v in zip(bars, values.values):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height(), f"{v:.4f}", ha="center", va="bottom", fontsize=8)
    ax.grid(axis="y", alpha=0.3)
    fig.tight_layout()
    fig.savefig(save_dir / f"final_{tag}_best_method_per_model_RMSE.png", dpi=150)
    plt.show()


def train_and_evaluate_split(split, read_cols, required_cols, feature_dict, input_label, best_params_by_combo):
    exp_id = int(split["Experiment"])
    exp_model_dir = MODEL_DIR / f"exp_{exp_id:02d}"
    exp_model_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 120)
    print(f"Final training/evaluation for Experiment {exp_id}")
    print("#" * 120)

    train_df, train_failures = load_scan_files(
        split["train_files"],
        read_cols=read_cols,
        required_cols=required_cols,
        cfg=cfg,
        desc=f"Final Exp {exp_id}: loading train scans",
        max_rows_total=FINAL_MAX_TRAIN_ROWS,
        max_rows_per_scan=cfg.max_rows_per_train_scan,
    )
    if len(train_failures):
        save_csv(train_failures, BENCHMARK_DIR / f"final_exp_{exp_id:02d}_train_failed_files.csv")

    y_train = train_df[cfg.target_col].to_numpy(dtype=np.float32)
    print(f"Final train shape: {train_df.shape}")

    specs = method_specs(cfg.random_state + exp_id)
    fitted_models = {}

    for model_name in MODEL_NAMES_TO_RUN:
        features = feature_dict[model_name]
        X_train = train_df[features]
        for method_name in METHOD_NAMES_TO_RUN:
            print("\n" + "-" * 100)
            print(f"Fitting final model | Exp {exp_id} | {model_name} × {method_name}")
            model = clone(specs[method_name]["estimator"])
            model.set_params(**best_params_by_combo[(model_name, method_name)])
            model.fit(X_train, y_train)

            fitted_models[(model_name, method_name)] = model
            save_pickle(
                {
                    "experiment": exp_id,
                    "model_name": model_name,
                    "method": method_name,
                    "input": input_label[model_name],
                    "features": features,
                    "best_params": best_params_by_combo[(model_name, method_name)],
                    "target_col": cfg.target_col,
                    "model": model,
                },
                exp_model_dir / f"{model_name}_{method_name}.pkl",
            )

        del X_train
        gc.collect()

    del train_df, y_train
    gc.collect()

    error_chunks = {combo: [] for combo in fitted_models}
    y_true_chunks = []
    test_failures_all = []

    for batch_i, batch_files in enumerate(make_batches(split["test_files"], cfg.test_batch_size_scans), start=1):
        test_df, test_failures = load_scan_files(
            batch_files,
            read_cols=read_cols,
            required_cols=required_cols,
            cfg=cfg,
            desc=f"Final Exp {exp_id}: loading test batch {batch_i}",
        )
        if len(test_failures):
            test_failures_all.append(test_failures)

        y_test = test_df[cfg.target_col].to_numpy(dtype=np.float32)
        y_true_chunks.append(y_test.copy())

        for (model_name, method_name), model in fitted_models.items():
            X_test = test_df[feature_dict[model_name]]
            pred = model.predict(X_test)
            error_chunks[(model_name, method_name)].append((y_test - pred).astype(np.float32))
            del X_test, pred

        del test_df, y_test
        gc.collect()

    if test_failures_all:
        save_csv(
            pd.concat(test_failures_all, ignore_index=True),
            BENCHMARK_DIR / f"final_exp_{exp_id:02d}_test_failed_files.csv",
        )

    y_true_all = np.concatenate(y_true_chunks) if y_true_chunks else np.array([], dtype=np.float32)

    rows = []
    for (model_name, method_name), chunks in error_chunks.items():
        err = np.concatenate(chunks) if chunks else np.array([], dtype=np.float32)
        m = metrics_from_error(err, threshold=cfg.vendor_target)
        denom = np.maximum(np.abs(y_true_all), 1e-6)
        m["MAPE"] = float(np.mean(np.abs(err) / denom) * 100) if err.size else np.nan
        rows.append({
            "Experiment": exp_id,
            "Model": model_name,
            "Method": method_name,
            "Input": input_label[model_name],
            "Feature_Count": len(feature_dict[model_name]),
            **m,
            "Best_Params": json.dumps(best_params_by_combo[(model_name, method_name)], ensure_ascii=False),
        })

    exp_result_df = pd.DataFrame(rows).sort_values(["Model", "RMSE"]).reset_index(drop=True)
    save_csv(exp_result_df, BENCHMARK_DIR / f"final_exp_{exp_id:02d}_performance.csv")

    plot_final_metric_comparison(exp_result_df, FIGURE_DIR, tag=f"exp_{exp_id:02d}")

    del fitted_models, error_chunks, y_true_chunks, y_true_all
    gc.collect()

    print("\nExperiment performance:")
    print(exp_result_df.to_string(index=False))

    return exp_result_df


# -----------------------------
# 1) 建立五個 feature models 與 rolling split
# -----------------------------
parquet_files = discover_parquet_files(cfg.align_dir)
print(f"Discovered parquet files: {len(parquet_files)}")

schema_df = inspect_sample_schema(parquet_files[0])
columns = schema_df["column"].tolist()

feature_dict, input_label, vibration_features, sensor_ids = build_feature_sets(columns, cfg)

# 僅保留這次要跑的五個模型，並檢查是否存在。
missing_models = [m for m in MODEL_NAMES_TO_RUN if m not in feature_dict]
if missing_models:
    raise ValueError(f"feature_dict 缺少這些模型：{missing_models}。請確認 cfg.include_model_v=True 且資料欄位完整。")
feature_dict = {m: feature_dict[m] for m in MODEL_NAMES_TO_RUN}
input_label = {m: input_label[m] for m in MODEL_NAMES_TO_RUN}

required_cols = sorted(set([cfg.target_col, "wavelength_nm", "x_pos", "y_pos"] + vibration_features))
read_cols = get_read_cols(columns, feature_dict, cfg)

sensor_availability = make_sensor_feature_availability(columns, sensor_ids, vibration_features)
save_csv(sensor_availability, BENCHMARK_DIR / "sensor_feature_availability.csv")

feature_settings = {
    "models": {
        model_name: {
            "input_label": input_label[model_name],
            "feature_count": len(feature_dict[model_name]),
            "features": feature_dict[model_name],
        }
        for model_name in MODEL_NAMES_TO_RUN
    },
    "vibration_feature_count": len(vibration_features),
    "gyro_feature_count": sum("_gyr_" in c for c in vibration_features),
    "sensor_ids": sensor_ids,
    "read_cols": read_cols,
    "required_cols": required_cols,
    "method_names": METHOD_NAMES_TO_RUN,
    "n_optuna_trials": N_OPTUNA_TRIALS,
}
save_json(feature_settings, BENCHMARK_DIR / "five_model_feature_settings.json")

print("Feature models:")
for model_name in MODEL_NAMES_TO_RUN:
    print(f"  {model_name:13s}: {len(feature_dict[model_name]):3d} features | {input_label[model_name]}")
print("Vibration feature count:", len(vibration_features))
print("Gyro feature count:", sum("_gyr_" in c for c in vibration_features))

profile_df, valid_files = profile_and_filter_scans(parquet_files, read_cols, required_cols, cfg)
splits = make_rolling_time_splits(valid_files, cfg)
if MAX_EXPERIMENTS_TO_RUN is not None:
    splits = splits[:MAX_EXPERIMENTS_TO_RUN]

print(f"Valid parquet scans: {len(valid_files)}")
print(f"Rolling experiments to run: {len(splits)}")


# -----------------------------
# 2) Optuna hyperparameter optimization
# -----------------------------
best_params_records = []
all_hpo_results = []
all_studies = []

if TUNE_ON_FIRST_SPLIT_ONLY:
    best_params_by_combo, hpo_results, studies = tune_all_model_methods_for_split(
        splits[0],
        read_cols,
        required_cols,
        feature_dict,
        random_state=cfg.random_state,
    )
    all_hpo_results.append(hpo_results)
    all_studies.append(studies)
    plot_optuna_history(hpo_results, FIGURE_DIR)

    for (model_name, method_name), params in best_params_by_combo.items():
        best_params_records.append({
            "HPO_Experiment": int(splits[0]["Experiment"]),
            "Model": model_name,
            "Method": method_name,
            "Input": input_label[model_name],
            "Feature_Count": len(feature_dict[model_name]),
            "Best_Params": json.dumps(params, ensure_ascii=False),
        })
else:
    # 這個模式會在每個 final experiment 前重新 tuning。
    # 因為非常耗時，所以預設不啟用。
    best_params_by_combo = None


# -----------------------------
# 3) Final train/test evaluation
# -----------------------------
final_results = []

for split in splits:
    exp_id = int(split["Experiment"])

    if not TUNE_ON_FIRST_SPLIT_ONLY:
        best_params_by_combo, hpo_results, studies = tune_all_model_methods_for_split(
            split,
            read_cols,
            required_cols,
            feature_dict,
            random_state=cfg.random_state + exp_id,
        )
        all_hpo_results.append(hpo_results)
        all_studies.append(studies)
        plot_optuna_history(hpo_results, FIGURE_DIR)

        for (model_name, method_name), params in best_params_by_combo.items():
            best_params_records.append({
                "HPO_Experiment": exp_id,
                "Model": model_name,
                "Method": method_name,
                "Input": input_label[model_name],
                "Feature_Count": len(feature_dict[model_name]),
                "Best_Params": json.dumps(params, ensure_ascii=False),
            })

    exp_result_df = train_and_evaluate_split(
        split,
        read_cols,
        required_cols,
        feature_dict,
        input_label,
        best_params_by_combo,
    )
    final_results.append(exp_result_df)


hpo_all_df = pd.concat(all_hpo_results, ignore_index=True)
best_params_df = pd.DataFrame(best_params_records)
final_all_df = pd.concat(final_results, ignore_index=True)

save_csv(hpo_all_df, BENCHMARK_DIR / "optuna_all_trials.csv")
save_csv(best_params_df, BENCHMARK_DIR / "best_hyperparameters_by_model_method.csv")
save_csv(final_all_df, BENCHMARK_DIR / "final_performance_by_experiment.csv")

summary_df = (
    final_all_df
    .groupby(["Model", "Method", "Input", "Feature_Count"], as_index=False)
    .agg(
        Experiments=("Experiment", "nunique"),
        N_Total=("N", "sum"),
        RMSE_mean=("RMSE", "mean"),
        RMSE_std=("RMSE", "std"),
        MAE_mean=("MAE", "mean"),
        MAE_std=("MAE", "std"),
        MAPE_mean=("MAPE", "mean"),
        Median_AE_mean=("Median_AE", "mean"),
        P90_AE_mean=("P90_AE", "mean"),
        P95_AE_mean=("P95_AE", "mean"),
        P99_AE_mean=("P99_AE", "mean"),
        Pct_error_gt_0p4dB_mean=("Pct_|error|>0.4dB", "mean"),
    )
    .sort_values(["Model", "RMSE_mean"])
    .reset_index(drop=True)
)

best_method_per_model_df = (
    summary_df
    .sort_values(["Model", "RMSE_mean"])
    .groupby("Model", as_index=False)
    .first()
    .sort_values("RMSE_mean")
    .reset_index(drop=True)
)

save_csv(summary_df, BENCHMARK_DIR / "final_performance_summary_by_model_method.csv")
save_csv(best_method_per_model_df, BENCHMARK_DIR / "final_best_method_per_model_summary.csv")

# 全部 experiments 的總比較圖。
plot_final_metric_comparison(final_all_df, FIGURE_DIR, tag="all_experiments")

print("\n" + "=" * 120)
print("Final benchmark summary by Model × Method, sorted within each Model by RMSE_mean")
print("=" * 120)
print(summary_df.to_string(index=False))

print("\n" + "=" * 120)
print("Best ML method for each of the five feature models")
print("=" * 120)
print(best_method_per_model_df.to_string(index=False))

print("\nOutput folder:")
print(BENCHMARK_DIR)
